# 26 — Bài báo, vòng sửa: bảng kết quả từ một nguồn, lần chạy chuẩn làm đại lượng chính (máy local)

**Phạm vi**

Thực thi amendment v4.3. Mọi bảng chính của bản thảo được sinh lại từ dự đoán đã lưu; không train lại, không gọi API, không tạo dự đoán mới, không đổi
ngưỡng, θ, prompt hay cấu hình.

**Đại lượng chính**

Lần chạy chuẩn của từng cấu hình: seed 42 với mô hình có huấn luyện, lần chạy duy nhất với LLM; hybrid và định tuyến dùng lần chạy chuẩn của thành phần.
Khoảng tin cậy 95% là bootstrap percentile ghép cặp 10.000 lần (seed 42). Trung bình ± độ lệch chuẩn ba seed chỉ có ở bảng phụ lục theo seed.

**Cổng nhất quán**

Mỗi số được sinh lại phải trùng từng ô với file của bước 2, 4 và 5 (sai số ≤ 1e-9). Bảng `paper_tables_gates.csv` ghi lại từng phép kiểm tra.

**Đầu ra**

Bảng `paper_tables_*.csv` với tên cột tiếng Anh như trong bản thảo, `paper_tables_columns.csv` (định nghĩa từng cột), `paper_tables_manifest.json`
(bảng → file nguồn, commit). Nhãn Hold-out, focused và Lazada chỉ được đọc lại để chấm lại dự đoán đã khóa. Không in văn bản review.

In [1]:
import os, sys, json, datetime, subprocess
import numpy as np, pandas as pd, yaml
from scipy.stats import spearmanr
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification.final_eval import load_frozen
from src.classification.evaluate import FOUR_CLASSES, OTHER_ID, metrics, cascade_labels
from src.classification import paper_analysis as PA
from src.classification import routing_calibration as RC
from src.classification import fair_controls as FC
from src.classification import cost_model as CM
from src.classification import paper_tables as PT
from src.classification.strategy_llm_fewshot.classify import cost_summary
from src.utils.llm import prices
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 300)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
PROMPTS = os.path.join(ROOT, '04_prompts', 'classification_fewshot')
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
DIRTY = bool(subprocess.run(['git', '-C', ROOT, 'status', '--porcelain', '02_src', '03_configs'], capture_output=True, text=True).stdout.strip())
FZ = load_frozen(ROOT)
SEEDS = FZ['tier1']['seeds']

def frozen_yaml(fname, key, must):
    txt = open(os.path.join(ROOT, '03_configs', 'frozen', fname), encoding='utf-8').read()
    assert txt.startswith('# FROZEN'), f'{fname}: thiếu header FROZEN'
    for m in must:
        assert m in txt, f'{fname}: thiếu dấu "{m}"'
    return yaml.safe_load(txt)[key]

TF = frozen_yaml('paper_tfidf_config.yaml', 'tfidf', ['amendment v3.2'])
NC = frozen_yaml('paper_neural_config.yaml', 'paper_neural', ['amendment v3.4', 'amendment v3.5'])
CFG_COST = frozen_yaml('paper_cost_model.yaml', 'cost_model', ['amendment v3.8'])
assert '| v4.3 |' in open(os.path.join(ROOT, '00_protocol', 'protocol_amendments.md'), encoding='utf-8').read()
SEL4 = json.load(open(os.path.join(RES, 'paper_step4_selection.json'), encoding='utf-8'))
SEL5 = json.load(open(os.path.join(RES, 'paper_step5_selection.json'), encoding='utf-8'))

def rd(name):
    return pd.read_csv(os.path.join(RES, name), encoding='utf-8-sig')

GATES, TABLES = [], {}
def log_gate(what, res):
    GATES.append({'check': what, 'keys compared': res[0], 'max abs deviation': res[1]})

def gate_pairs(new, old, what, old_diff='chênh A − B (seed 42)'):
    m = new.merge(old, left_on=['key A', 'key B'], right_on=['A', 'B'], suffixes=('', '_old'))
    assert len(m) == len(new) == len(old), (what, len(m), len(new), len(old))
    dev = max(float(np.abs(m['Δ Macro-F1 (A − B)'] * 100 - m[old_diff]).max()),
              float(np.abs(m['CI low'] * 100 - m['CI 2,5%']).max()), float(np.abs(m['CI high'] * 100 - m['CI 97,5%']).max()),
              float(np.abs(m['p (exact)'] - m['p (exact)_old']).max()), float(np.abs(m['p (Holm)'] - m['p (Holm)_old']).max()))
    assert dev <= 1e-7, (what, dev)
    GATES.append({'check': what, 'keys compared': len(m), 'max abs deviation': dev})

def table(name, df, sources, note, pct=True):
    TABLES[name] = {'df': df, 'sources': sources, 'note': note, 'pct': pct}

print('Ngày chạy:', RUN_DATE, '| commit:', COMMIT, '| thay đổi chưa commit ở 02_src/03_configs:', DIRTY, '| seeds:', SEEDS)

Ngày chạy: 21/11/2026 | commit: b436fad136189c47383eb98f13dfdf1539e9c940 | thay đổi chưa commit ở 02_src/03_configs: True | seeds: [42, 1337, 2024]


## Kiểm toán dữ liệu và cấu hình (A3)

**Nội dung**

ID các tập không giao nhau; số lượng từng lớp; trùng y hệt và gần trùng với Train ∪ Validation; ví dụ few-shot thuộc Train và số ví dụ độc nhất.

In [2]:
SPLITS = {'Train': 'shopee_train.csv', 'Validation': 'shopee_validation.csv', 'Shopee hold-out': 'shopee_holdout.csv',
          'Shopee focused': 'shopee_focused_feature_ux.csv', 'Lazada': 'lazada_external_test.csv'}
ALL = {k: pd.read_csv(os.path.join(DATA, f), encoding='utf-8-sig') for k, f in SPLITS.items()}
OVER = PT.id_overlaps(ALL)
assert (OVER['Shared review IDs'] == 0).all(), OVER
COUNTS = PT.class_counts(ALL)
REF = pd.concat([ALL['Train']['text'], ALL['Validation']['text']]).astype(str)
ND = {k: PA.near_duplicates(REF, ALL[k]['text'].astype(str), FC.NEAR_MAIN) for k in ('Shopee hold-out', 'Shopee focused', 'Lazada')}
DUP = {k: (int(v['exact'].sum()), int(v['near'].sum()), int(((v['max_cosine'] >= FC.NEAR_SENS) & ~v['exact']).sum())) for k, v in ND.items()}
for i, col in enumerate(['Exact duplicates of Train ∪ Validation', 'Near duplicates (cosine ≥ 0.90)', 'Near duplicates (cosine ≥ 0.80)']):
    COUNTS[col] = COUNTS['Set'].map(lambda s: DUP[s][i] if s in DUP else np.nan)
old = rd('paper_step5_near_duplicates_summary.csv')
SETMAP = {'holdout': 'Shopee hold-out', 'lazada': 'Lazada', 'focused': 'Shopee focused'}
for _, r in old.iterrows():
    i = {'trùng y hệt': 0, 'gần trùng ≥ 0,90': 1, 'gần trùng ≥ 0,80': 2}[r['loại']]
    assert DUP[SETMAP[r['tập']]][i] == r['số review'], (r['tập'], r['loại'])
assert int((~PA.keep_mask(ALL['Shopee hold-out']['text'].astype(str), REF)).sum()) == DUP['Shopee hold-out'][0] == 217
GATES.append({'check': 'trùng y hệt và gần trùng = paper_step5_near_duplicates_summary; 217 trùng y hệt ở Hold-out', 'keys compared': len(old), 'max abs deviation': 0.0})
print(COUNTS.to_string(index=False)); print(OVER.to_string(index=False))

TRAIN_IDS = set(ALL['Train']['review_id'])
OTHER_IDS = set().union(*[set(ALL[k]['review_id']) for k in SPLITS if k != 'Train'])
FS, FS_SUM = PT.fewshot_audit(PROMPTS, TRAIN_IDS, OTHER_IDS)
assert FS['All in Train'].all() and FS['In another split'].sum() == 0
V1 = FS_SUM.set_index('Prompt version').loc['v1']
L_DISTINCT = len(ALL['Validation']) + int(V1['Distinct exemplars, all prompts'])
L_PER_PROMPT = len(ALL['Validation']) + int(FS[(FS['Prompt version'] == 'v1') & FS['Prompt'].isin(['tier1', 'tier2'])]['Exemplars'].sum())
print('\nVí dụ few-shot:'); print(FS.drop(columns=['Selection rule']).to_string(index=False)); print(FS_SUM.to_string(index=False))
print(f"Nhãn ban đầu của phương án chỉ LLM: cấu hình v3.8 = {CFG_COST['labels_initial']['llm_only']}; đếm theo prompt = {L_PER_PROMPT}; review độc nhất = {L_DISTINCT}")
table('m1_data_splits', COUNTS, ['01_data/processed/*.csv', 'paper_step5_near_duplicates_summary.csv'], 'Class distribution of every split and duplicates of Train ∪ Validation.', pct=True)
table('m2_split_id_overlaps', OVER, ['01_data/processed/*.csv'], 'Shared review IDs between splits (all zero).', pct=False)
table('m3_fewshot_exemplars', FS, ['04_prompts/classification_fewshot/*_exemplars_*.json'], 'Few-shot exemplars of every prompt: count, classes, membership in Train, selection rule.', pct=False)
table('m3b_fewshot_summary', FS_SUM, ['04_prompts/classification_fewshot/*_exemplars_*.json'], 'Distinct few-shot exemplars across prompts.', pct=False)

            Set    n  BUG  FEATURE  UX  OTHER  Actionable  Actionable (%)  Exact duplicates of Train ∪ Validation  Near duplicates (cosine ≥ 0.90)  Near duplicates (cosine ≥ 0.80)
          Train 1200   75      129 132    864         336        0.280000                                     NaN                              NaN                              NaN
     Validation  350   27       22  22    279          71        0.202857                                     NaN                              NaN                              NaN
Shopee hold-out 1000   89       24  70    817         183        0.183000                                   217.0                             17.0                             48.0
 Shopee focused  150   10       56  25     59          91        0.606667                                     1.0                              2.0                              5.0
         Lazada  500   39       17  25    419          81        0.162000                           

## Dự đoán đã lưu và các cấu hình

**Nguồn**

Bước 2 (28 cấu hình), bước 4 (định tuyến theo độ tin cậy, hiệu chỉnh), bước 5 (phẳng có ngưỡng, các chính sách định tuyến). Xác suất phẳng phải cho đúng
nhãn argmax đã báo cáo; router θ_max và θ_98 của bước 4 và bước 5 phải trùng nhau.

In [3]:
NAME = {'holdout': 'Shopee hold-out', 'lazada': 'Lazada', 'focused': 'Shopee focused'}
D = {s: ALL[n] for s, n in NAME.items()}
IDS = {s: d['review_id'].tolist() for s, d in D.items()}
Y4 = {s: d['gold_label'].map(RC.LBL).to_numpy() for s, d in D.items()}
PRED = {s: PA.step2_predictions(RES, s, IDS[s], FZ, TF, NC) for s in NAME}
CFG, SKIP = {}, {}
for s in NAME:
    CFG[s], SKIP[s] = PA.configurations_step2(*PRED[s])
assert len(CFG['holdout']) == len(CFG['lazada']) == 28
LLM = {s: PA.canonical(PRED[s][2][PA.LLM]) for s in NAME}
FLAT = {s: {x: FC.flat_probs(RES, s, IDS[s], m, SEEDS) for x, m in FC.TRAINED.items()} for s in NAME}
for s in NAME:
    for x in FC.TRAINED:
        assert all((FLAT[s][x][k].argmax(1) == PRED[s][2][x][k]).all() for k in SEEDS), (s, x)
R4, MASK5, R5 = {}, {}, {}
for s in ('holdout', 'lazada'):
    small = {x: RC.probabilities(RES, s, IDS[s], 'flat', m, SEEDS) for x, m in RC.SMALL.items()}
    cfg, rate = RC.routed_configs(small, LLM[s], SEL4['routing'])
    R4[s] = {'cfg': cfg, 'rate': rate}
    MASK5[s] = PT.b2_masks(FLAT[s]['PhoBERT'], FLAT[s]['ViSoBERT'], SEL5['routing_v41'], SEL5['budgets'], SEEDS)
    R5[s] = PT.routed(FLAT[s]['PhoBERT'], LLM[s], MASK5[s])
    for n in (PT.B2_CONF_MAX, PT.B2_CONF_98):
        assert all((R5[s][n][k] == cfg[n][k]).all() and (MASK5[s][n][k] == rate[n][k]).all() for k in SEEDS), (s, n)
print({s: len(CFG[s]) for s in NAME}, '| hybrid không chấm:', {s: [(d['tầng 1'], d['tầng 2'], d['review thiếu tầng 2']) for d in SKIP[s]] for s in NAME})

def parts(name):
    if name.startswith('Phẳng '):
        return name[len('Phẳng '):], '—'
    if name.startswith('Cascade '):
        return (name[len('Cascade '):],) * 2
    return tuple(name.split(' → '))

rows = []
for name, runs in CFG['holdout'].items():
    a, b = parts(name)
    rows.append({'Configuration': PT.en(name), 'Architecture': PT.architecture(name), 'Tier 1 or flat model': PT.en(a), 'Tier 2 model': PT.en(b),
                 'Runs': len(runs), 'Amendment': 'v3.2–v3.5 (joint scoring)', 'Status': 'scored'})
gap = {s: {(d['tầng 1'], d['tầng 2']): d['review thiếu tầng 2'] for d in SKIP[s]} for s in ('holdout', 'lazada')}
for (a, b), n in gap['holdout'].items():
    rows.append({'Configuration': PT.en(f'{a} → {b}'), 'Architecture': 'hybrid', 'Tier 1 or flat model': PT.en(a), 'Tier 2 model': PT.en(b), 'Runs': 0,
                 'Amendment': 'v3.2–v3.5 (joint scoring)',
                 'Status': f"not scored: the LLM tier 2 was run only on reviews kept by the ViSoBERT, Bi-LSTM or LLM tier 1, so {n} hold-out and {gap['lazada'][(a, b)]} Lazada reviews kept by the {PT.en(a)} tier 1 have no tier-2 label"})
for x in FC.TRAINED:
    rows.append({'Configuration': PT.en(f'Phẳng {x} (ngưỡng)'), 'Architecture': 'flat (thresholded)', 'Tier 1 or flat model': PT.en(x), 'Tier 2 model': '—',
                 'Runs': 3, 'Amendment': 'v4.2 (B1)', 'Status': 'scored, exploratory'})
for n in R4['holdout']['cfg']:
    rows.append({'Configuration': PT.en(n), 'Architecture': 'routing', 'Tier 1 or flat model': PT.en(n.split(' ⇢ ')[0]), 'Tier 2 model': 'Flat LLM',
                 'Runs': 3, 'Amendment': 'v4.1', 'Status': 'scored, exploratory'})
for n in R5['holdout']:
    if n not in R4['holdout']['cfg']:
        rows.append({'Configuration': PT.en(n), 'Architecture': 'routing', 'Tier 1 or flat model': 'PhoBERT', 'Tier 2 model': 'Flat LLM',
                     'Runs': 3, 'Amendment': 'v4.2 (B2)', 'Status': 'scored, exploratory'})
for x in RC.TIER1:
    rows.append({'Configuration': PT.en(f'Cascade {x} (hiệu chỉnh)'), 'Architecture': 'cascade (calibrated)', 'Tier 1 or flat model': PT.en(x),
                 'Tier 2 model': PT.en(x), 'Runs': 3, 'Amendment': 'v4.1', 'Status': 'scored, exploratory'})
CONFIGS = pd.DataFrame(rows)
print(CONFIGS.groupby(['Amendment', 'Architecture', 'Status']).size().to_string())
table('m4_configurations', CONFIGS, ['paper_step2_*', 'paper_step4_*', 'paper_step5_*'], 'Every configuration: 28 of the joint scoring (5 cascades, 5 flat, 18 hybrids), 2 hybrids not scored and the exploratory additions of v4.1 and v4.2.', pct=False)

{'holdout': 28, 'lazada': 28, 'focused': 21} | hybrid không chấm: {'holdout': [('TF-IDF', 'LLM few-shot', 17), ('PhoBERT', 'LLM few-shot', 28)], 'lazada': [('TF-IDF', 'LLM few-shot', 15), ('PhoBERT', 'LLM few-shot', 26)], 'focused': [('TF-IDF', 'LLM few-shot', 37), ('Bi-LSTM + fastText', 'LLM few-shot', 30), ('ViSoBERT', 'LLM few-shot', 30), ('PhoBERT', 'LLM few-shot', 38)]}
Amendment                  Architecture          Status                                                                                                                                                                              
v3.2–v3.5 (joint scoring)  cascade               scored                                                                                                                                                                                   5
                           flat                  scored                                                                                                     

## RQ1 — chất lượng trong và ngoài ứng dụng

**Bảng**

Mười cấu hình thuần trên Hold-out và Lazada; chuyển ứng dụng (Lazada − Hold-out); kết quả từng tầng; focused set (mô tả); so sánh ghép cặp chính trong họ
end-to-end 378 cặp đã đăng ký; Hold-out bỏ trùng; biến thể prompt trung tính; mã hóa thủ công lỗi giữ nhầm OTHER.

In [4]:
E2E, DIFF, YY, CANON, MC = {}, {}, {}, {}, {}
for s in NAME:
    E2E[s], DIFF[s], YY[s], CANON[s] = PT.e2e_table(Y4[s], CFG[s])
for s, tag in (('holdout', 'Hold-out'), ('lazada', 'Lazada')):
    log_gate(f'end-to-end {s}: Macro-F1 và CI = paper_step2_e2e_{s}_full',
             PT.gate(E2E[s], rd(f'paper_step2_e2e_{s}_full.csv'), {'Macro-F1': 'Macro-F1 chuẩn', 'CI low': 'CI 2,5%', 'CI high': 'CI 97,5%'}, what=s))
    old = rd(f'paper_step2_e2e_{s}_diff.csv')
    m = DIFF[s].merge(old, on=['A', 'B'], suffixes=('', '_old'))
    assert len(m) == len(old) == len(DIFF[s]) == 378
    dev = max(float(np.abs(m[c] - m[c + '_old']).max()) for c in ('chênh (A − B)', 'CI 2,5%', 'CI 97,5%'))
    assert dev <= PT.GATE_TOL
    GATES.append({'check': f'end-to-end {s}: 378 chênh lệch ghép cặp = paper_step2_e2e_{s}_diff', 'keys compared': len(m), 'max abs deviation': dev})
    fam = f'end-to-end – {tag}'
    MC[s] = PA.mcnemar_table(YY[s], CANON[s], fam)
    old = rd('paper_step2_mcnemar.csv')
    m = MC[s].merge(old[old['họ so sánh'] == fam], on=['A', 'B'], suffixes=('', '_old'))
    assert len(m) == 378
    dev = max(float(np.abs(m[c] - m[c + '_old']).max()) for c in ('p (exact)', 'p (Holm)'))
    assert dev <= 1e-12
    GATES.append({'check': f'McNemar {fam}: p exact và Holm = paper_step2_mcnemar', 'keys compared': len(m), 'max abs deviation': dev})

PURE = [n for x in PA.ORDER for n in (f'Phẳng {x}', f'Cascade {x}')]
MAIN = ['Configuration', 'Architecture', 'n', 'Macro-F1', 'CI low', 'CI high', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER',
        'Precision ACT', 'Recall ACT', 'Forwarded']

def rows_of(tab, keys, s, cols=MAIN):
    t = tab.set_index('key')
    keys = [k for k in keys if k in t.index]
    out = t.loc[keys].reset_index()[['key'] + cols]
    out.insert(1, 'Set', NAME[s])
    return out

RQ1_E2E = pd.concat([rows_of(E2E[s], PURE, s) for s in ('holdout', 'lazada')], ignore_index=True)
print(PT.pct(PT.drop_keys(RQ1_E2E), ['Macro-F1', 'CI low', 'CI high', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER', 'Precision ACT', 'Recall ACT', 'Forwarded']).to_string(index=False))
table('rq1_e2e', RQ1_E2E, ['paper_pred_*', 'gd10_pred_*', 'paper_flat_llm_*', 'gd10_llm_*'], 'End-to-end four-class results of the ten pure configurations on the Shopee hold-out and Lazada (canonical run).')
FOCUSED = rows_of(E2E['focused'], PURE, 'focused')
table('rq1_focused_e2e', FOCUSED, ['paper_pred_*', 'gd10_pred_*', 'paper_flat_llm_focused.csv'], 'End-to-end results on the FEATURE/UX-enriched Shopee focused set (descriptive; LLM tier 1 was not run on this set).')

            Set    Configuration Architecture    n  Macro-F1  CI low  CI high  F1 BUG  F1 FEATURE  F1 UX  F1 OTHER  Precision ACT  Recall ACT  Forwarded
Shopee hold-out      Flat TF-IDF         flat 1000      62.8    57.5     67.6    47.7        28.6   81.9      92.8           68.6        65.6       17.5
Shopee hold-out   Cascade TF-IDF      cascade 1000      61.3    56.9     65.5    53.2        21.3   79.4      91.4           60.0        78.7       24.0
Shopee hold-out     Flat Bi-LSTM         flat 1000      62.9    56.4     68.7    46.6        33.3   78.4      93.3           73.4        61.7       15.4
Shopee hold-out  Cascade Bi-LSTM      cascade 1000      54.2    49.9     58.3    42.6        18.6   67.9      87.8           49.7        83.1       30.6
Shopee hold-out    Flat ViSoBERT         flat 1000      66.7    60.3     72.1    46.8        45.5   80.3      94.2           82.1        60.1       13.4
Shopee hold-out Cascade ViSoBERT      cascade 1000      62.9    57.6     67.8    4

In [5]:
T1 = {s: PT.tier1_table(Y4[s], PRED[s][0]) for s in ('holdout', 'lazada')}
T2 = {s: PT.tier2_table(Y4[s], PRED[s][1]) for s in ('holdout', 'focused')}
log_gate('tầng 1 Hold-out: F1 và recall ACT = paper_step2_tier1_holdout',
         PT.gate(T1['holdout'], rd('paper_step2_tier1_holdout.csv'), {'F1 ACT': 'f1 chuẩn', 'Recall ACT': 'recall chuẩn'}, key_old='chiến lược', what='t1'))
for s in ('holdout', 'focused'):
    log_gate(f'tầng 2 {s}: Macro-F1 = paper_step2_tier2_{s}',
             PT.gate(T2[s], rd(f'paper_step2_tier2_{s}.csv'), {'Macro-F1': 'Macro-F1 chuẩn'}, key_old='chiến lược', what=f't2 {s}'))
yh = Y4['holdout']; g = yh != OTHER_ID
c1 = {k: np.asarray(PA.canonical(v)).astype(int) for k, v in PRED['holdout'][0].items()}
c2 = {k: np.asarray(PA.canonical(v))[g] for k, v in PRED['holdout'][1].items()}
MC_T = pd.concat([PA.mcnemar_table((yh != OTHER_ID).astype(int), c1, 'tầng 1 – Hold-out'), PA.mcnemar_table(yh[g], c2, 'tầng 2 – Hold-out')], ignore_index=True)
old = rd('paper_step2_mcnemar.csv')
m = MC_T.merge(old, on=['họ so sánh', 'A', 'B'], suffixes=('', '_old'))
assert len(m) == len(MC_T) == 20
dev = max(float(np.abs(m[c] - m[c + '_old']).max()) for c in ('p (exact)', 'p (Holm)'))
assert dev <= 1e-12
GATES.append({'check': 'McNemar tầng 1 và tầng 2 Hold-out = paper_step2_mcnemar', 'keys compared': len(m), 'max abs deviation': dev})

TIER1 = pd.concat([t.assign(Set=NAME[s]) for s, t in T1.items()], ignore_index=True)
TIER2 = pd.concat([t.assign(Set=NAME[s]) for s, t in T2.items()], ignore_index=True)
TIER_MC = MC_T.rename(columns={'họ so sánh': 'Family', 'A đúng, B sai': 'A right, B wrong', 'A sai, B đúng': 'A wrong, B right'})
TIER_MC['Family'] = TIER_MC['Family'].map({'tầng 1 – Hold-out': 'Tier 1, Shopee hold-out (10 pairs)', 'tầng 2 – Hold-out': 'Tier 2, Shopee hold-out (10 pairs)'})
TIER_MC['A'], TIER_MC['B'] = TIER_MC['A'].map(PT.en), TIER_MC['B'].map(PT.en)
TIER_MC = TIER_MC[['Family', 'A', 'B', 'A right, B wrong', 'A wrong, B right', 'p (exact)', 'p (Holm)']]
print(PT.pct(PT.drop_keys(TIER1), ['F1 ACT', 'F1 ACT CI low', 'F1 ACT CI high', 'Recall ACT', 'Recall ACT CI low', 'Recall ACT CI high', 'Precision ACT', 'Precision ACT CI low', 'Precision ACT CI high', 'Macro-F1 (2 classes)', 'Forwarded']).to_string(index=False))
print(PT.pct(PT.drop_keys(TIER2), ['Macro-F1', 'CI low', 'CI high', 'F1 BUG', 'F1 FEATURE', 'F1 UX']).to_string(index=False))
table('rq1_tier1', TIER1, ['paper_pred_*', 'gd10_pred_*', 'gd10_llm_*'], 'Tier 1 (ACTIONABLE vs NOT_ACTIONABLE) on the Shopee hold-out and Lazada, canonical run with 95% CIs.')
table('rq1_tier2', TIER2, ['paper_pred_*', 'gd10_pred_*', 'gd10_llm_*'], 'Tier 2 (BUG/FEATURE/UX) on gold-actionable reviews of the Shopee hold-out (183) and the focused set, canonical run with 95% CI.')
table('rq1_tier_mcnemar', TIER_MC, ['paper_step2_mcnemar.csv'], 'Exact McNemar tests between strategies at tier 1 and tier 2 (Holm within each tier family).', pct=False)

P1 = [('Phẳng LLM few-shot', 'Phẳng PhoBERT'), ('Cascade LLM few-shot', 'Phẳng LLM few-shot'), ('Cascade LLM few-shot', 'Phẳng PhoBERT'),
      ('Phẳng PhoBERT', 'Phẳng ViSoBERT'), ('Phẳng PhoBERT', 'Phẳng TF-IDF'), ('Phẳng PhoBERT', 'Phẳng Bi-LSTM + fastText'),
      ('Phẳng ViSoBERT', 'Phẳng TF-IDF'), ('Phẳng TF-IDF', 'Phẳng Bi-LSTM + fastText'), ('Phẳng TF-IDF', 'Cascade ViSoBERT')]
FAM_E2E = {s: f'End-to-end, {NAME[s]} (378 pairs)' for s in ('holdout', 'lazada')}
RQ1_PAIRS = pd.concat([PT.pair_table(YY[s], DIFF[s], CANON[s], P1, FAM_E2E[s], holm_from=MC[s]) for s in ('holdout', 'lazada')], ignore_index=True)
ALL_PAIRS = pd.concat([PT.pair_table(YY[s], DIFF[s], CANON[s], list(zip(MC[s]['A'], MC[s]['B'])), FAM_E2E[s], holm_from=MC[s]) for s in ('holdout', 'lazada')], ignore_index=True)
v = PT.pct(PT.drop_keys(RQ1_PAIRS), ['Δ Macro-F1 (A − B)', 'CI low', 'CI high'])
print(v.round({'p (exact)': 4, 'p (Holm)': 4}).to_string(index=False))
table('rq1_pairs', RQ1_PAIRS, ['paper_step2_e2e_*_diff.csv', 'paper_step2_mcnemar.csv'], 'Main pairwise comparisons of RQ1: paired bootstrap difference of Macro-F1 with 95% CI and exact McNemar, Holm over the registered 378-pair family.')
table('app_pairs_all', ALL_PAIRS, ['paper_step2_e2e_*_diff.csv', 'paper_step2_mcnemar.csv'], 'Supplement: all 378 pairwise comparisons of the 28 configurations on each set.')

Strategy    n  F1 ACT  F1 ACT CI low  F1 ACT CI high  Recall ACT  Recall ACT CI low  Recall ACT CI high  Precision ACT  Precision ACT CI low  Precision ACT CI high  Macro-F1 (2 classes)  Forwarded             Set
  TF-IDF 1000    68.1           62.7            73.0        78.7               72.7                84.4           60.0                  53.8                   66.2                  79.8       24.0 Shopee hold-out
 Bi-LSTM 1000    62.2           57.1            67.0        83.1               77.4                88.4           49.7                  44.2                   55.2                  75.0       30.6 Shopee hold-out
ViSoBERT 1000    64.9           59.2            70.2        67.8               61.0                74.4           62.3                  55.6                   69.0                  78.3       19.9 Shopee hold-out
 PhoBERT 1000    65.7           60.3            70.8        78.1               72.0                83.9           56.7                  50.6        

                                 Family             A                B  Δ Macro-F1 (A − B)  CI low  CI high  CI contains 0  A right, B wrong  A wrong, B right  p (exact)  p (Holm)  Holm family size
End-to-end, Shopee hold-out (378 pairs)      Flat LLM     Flat PhoBERT                 9.2     3.8     14.8          False                73                59     0.2578    1.0000               378
End-to-end, Shopee hold-out (378 pairs)   Cascade LLM         Flat LLM                 1.3    -2.0      4.6           True                35                18     0.0270    1.0000               378
End-to-end, Shopee hold-out (378 pairs)   Cascade LLM     Flat PhoBERT                10.5     4.4     16.8          False                77                46     0.0066    0.9491               378
End-to-end, Shopee hold-out (378 pairs)  Flat PhoBERT    Flat ViSoBERT                 4.1    -1.2      9.5           True                35                35     1.0000    1.0000               378
End-to-end

In [6]:
h = 'holdout'
KEEP_EX = PA.keep_mask(D[h]['text'].astype(str), REF)
nd = ND['Shopee hold-out']
KEEP_NR = ~(nd['exact'] | nd['near']).to_numpy()
assert KEEP_EX.sum() == 783 and KEEP_NR.sum() == 766
DCFG = {**{k: CFG[h][k] for k in PURE}, **{k: R4[h]['cfg'][k] for k in (PT.B2_CONF_MAX, PT.B2_CONF_98)}}
full = PT.e2e_table(Y4[h], DCFG)[0].set_index('key')
ex = PT.e2e_table(Y4[h], DCFG, mask=KEEP_EX)[0].set_index('key')
nr = PT.e2e_table(Y4[h], DCFG, mask=KEEP_NR)[0].set_index('key')
keys = list(DCFG)
DEDUP = pd.DataFrame({'key': keys, 'Configuration': [PT.en(k) for k in keys],
                      'Macro-F1, full hold-out (n = 1,000)': full.loc[keys, 'Macro-F1'].values,
                      'Macro-F1 without exact duplicates (n = 783)': ex.loc[keys, 'Macro-F1'].values,
                      'CI low (without exact)': ex.loc[keys, 'CI low'].values, 'CI high (without exact)': ex.loc[keys, 'CI high'].values,
                      'Macro-F1 without exact and near duplicates (n = 766)': nr.loc[keys, 'Macro-F1'].values,
                      'CI low (without exact and near)': nr.loc[keys, 'CI low'].values, 'CI high (without exact and near)': nr.loc[keys, 'CI high'].values})
DEDUP['Δ without exact'] = DEDUP['Macro-F1 without exact duplicates (n = 783)'] - DEDUP['Macro-F1, full hold-out (n = 1,000)']
DEDUP['Δ without exact and near'] = DEDUP['Macro-F1 without exact and near duplicates (n = 766)'] - DEDUP['Macro-F1, full hold-out (n = 1,000)']
log_gate('Hold-out bỏ trùng y hệt và gần trùng = paper_step5_near_duplicates_holdout_e2e',
         PT.gate(DEDUP, rd('paper_step5_near_duplicates_holdout_e2e.csv'), {'Macro-F1 without exact and near duplicates (n = 766)': 'Macro-F1 bỏ trùng và gần trùng (seed 42)'}, scale=100, what='dedup'))
RHO = {'exact': spearmanr(DEDUP.iloc[:, 2], DEDUP.iloc[:, 3]).correlation, 'exact+near': spearmanr(DEDUP.iloc[:, 2], DEDUP.iloc[:, 6]).correlation}
print('Spearman thứ hạng (đầy đủ so với bỏ trùng):', {k: round(v, 3) for k, v in RHO.items()})
print(PT.pct(PT.drop_keys(DEDUP), [c for c in DEDUP.columns if c not in ('key', 'Configuration')]).to_string(index=False))
table('rq1_dedup', DEDUP, ['paper_step2_dedup_holdout.csv', 'paper_step5_near_duplicates_holdout_e2e.csv'], f"Shopee hold-out after removing reviews whose normalised text occurs in Train ∪ Validation (217) and near duplicates (cosine ≥ 0.90, 17); Spearman rank correlation with the full hold-out {RHO['exact']:.3f} and {RHO['exact+near']:.3f}.")

PVN = {'Cascade LLM – v1': 'Cascade LLM, prompt v1', 'Phẳng LLM – v1': 'Flat LLM, prompt v1', 'Cascade LLM – v1n': 'Cascade LLM, prompt v1n (app-neutral)', 'Phẳng LLM – v1n': 'Flat LLM, prompt v1n (app-neutral)'}
PV = rd('paper_neutral_lazada_e2e_scores.csv')
lz = E2E['lazada'].set_index('key')
assert round(lz.loc['Cascade LLM few-shot', 'Macro-F1'] * 100, 1) == PV.set_index('cấu hình').loc['Cascade LLM – v1', 'Macro-F1 (4 lớp)']
assert round(lz.loc['Phẳng LLM few-shot', 'Macro-F1'] * 100, 1) == PV.set_index('cấu hình').loc['Phẳng LLM – v1', 'Macro-F1 (4 lớp)']
GATES.append({'check': 'biến thể prompt Lazada: dòng v1 = Macro-F1 lần chạy chuẩn của LLM trên Lazada (làm tròn 0,1)', 'keys compared': 2, 'max abs deviation': 0.0})
PV = PV.rename(columns={'cấu hình': 'Configuration', 'Macro-F1 (4 lớp)': 'Macro-F1 (%)', 'F1 BUG': 'F1 BUG (%)', 'F1 FEATURE': 'F1 FEATURE (%)', 'F1 UX': 'F1 UX (%)',
                        'F1 OTHER': 'F1 OTHER (%)', 'Recall ACT': 'Recall ACT (%)', 'Precision ACT': 'Precision ACT (%)'})
PV['Configuration'] = PV['Configuration'].map(PVN)
PVD = rd('paper_neutral_lazada_diff_scores.csv')
PVD = pd.DataFrame({'Comparison': PVD['so sánh'].map(lambda t: ' − '.join(PVN[x.strip()] for x in t.split(' − '))), 'Δ Macro-F1 (points)': PVD['chênh Macro-F1'],
                    'CI low (points)': PVD['CI 2,5%'], 'CI high (points)': PVD['CI 97,5%'], 'CI contains 0': PVD['CI chứa 0'], 'p (Holm)': PVD['McNemar p (Holm)']})
table('rq1_prompt_variant', PV, ['paper_neutral_lazada_e2e_scores.csv'], 'Original prompt v1 against the app-neutral prompt v1n on Lazada (single LLM runs, GPT-4o-mini).', pct=False)
table('rq1_prompt_variant_pairs', PVD, ['paper_neutral_lazada_diff_scores.csv', 'paper_neutral_lazada_mcnemar_scores.csv'], 'Differences v1n − v1 with paired bootstrap 95% CI and McNemar p (Holm over the six pairs of the variant family).', pct=False)

CAT = {'giao hàng': 'Delivery', 'người bán, sản phẩm': 'Seller, product', 'giá, phí, voucher': 'Price, fees, vouchers',
       'chính sách, tài khoản, đơn hàng': 'Policy, account, orders', 'chăm sóc khách hàng': 'Customer service', 'than phiền app chung chung': 'Generic app complaint',
       'khen chung chung': 'Generic praise', 'khác': 'Other', 'tổng': 'Total'}
def coding_cols(c):
    if c == 'nhóm chính':
        return 'Main category'
    arch = 'Cascade LLM' if c.startswith('cascade') else 'Flat LLM'
    if 'chênh' in c:
        return f'{arch}, Lazada − hold-out (points of OTHER reviews)'
    st = 'hold-out' if 'Hold-out' in c else 'Lazada'
    return f'{arch}, {st} (% of OTHER reviews)' if '% OTHER' in c else f'{arch}, {st} (% of coded errors)'
HC = rd('paper_lazada_errors_human_coding.csv'); HC = HC.rename(columns={c: coding_cols(c) for c in HC.columns}); HC['Main category'] = HC['Main category'].map(CAT)
CT = rd('paper_lazada_errors_contribution.csv'); CT = CT.rename(columns={c: coding_cols(c) for c in CT.columns}); CT['Main category'] = CT['Main category'].map(CAT)
table('rq1_error_coding', HC, ['paper_lazada_errors_human_coding.csv', '07_human_eval/paper_lazada_errors/'], 'Manual coding of OTHER reviews that the LLM labelled actionable (share of each model’s coded errors).', pct=False)
table('rq1_error_contribution', CT, ['paper_lazada_errors_contribution.csv'], 'Contribution of each category to the false-actionable rate on gold-OTHER reviews; policy, account and order complaints explain 9.0 of the 9.5 points (cascade) of the Lazada − hold-out gap.', pct=False)

Spearman thứ hạng (đầy đủ so với bỏ trùng): {'exact': np.float64(0.979), 'exact+near': np.float64(0.979)}
       Configuration  Macro-F1, full hold-out (n = 1,000)  Macro-F1 without exact duplicates (n = 783)  CI low (without exact)  CI high (without exact)  Macro-F1 without exact and near duplicates (n = 766)  CI low (without exact and near)  CI high (without exact and near)  Δ without exact  Δ without exact and near
         Flat TF-IDF                                 62.8                                         62.3                    57.0                     67.3                                                  62.0                             56.7                              67.1             -0.5                      -0.7
      Cascade TF-IDF                                 61.3                                         60.4                    55.9                     64.6                                                  60.1                             55.5                        

## RQ2 — kiến trúc và chính sách định tuyến trong cùng điều kiện so sánh

**Bảng**

Phẳng argmax, phẳng có ngưỡng và cascade của từng chiến lược (B1); các chính sách định tuyến sang LLM (v4.1 và B2) kèm đối chứng ngẫu nhiên cùng số lượt gọi;
chênh lệch ghép cặp trong đúng họ Holm đã đăng ký.

In [7]:
THR = {x: SEL5['b1'][x]['threshold'] for x in FC.TRAINED}
B1 = {}
for s in NAME:
    cfg = PT.b1_configs(CFG[s], FLAT[s], THR)
    tab, diff, y, canon = PT.e2e_table(Y4[s], cfg)
    B1[s] = {'cfg': cfg, 'tab': tab, 'diff': diff, 'y': y, 'canon': canon}
    old = rd('paper_step5_b1_results.csv'); old = old[old['tập'] == s]
    log_gate(f'B1 {s}: Macro-F1, CI, recall ACT, tỷ lệ chuyển = paper_step5_b1_results',
             PT.gate(tab, old, {'Macro-F1': 'Macro-F1 chuẩn', 'CI low': 'CI 2,5%', 'CI high': 'CI 97,5%', 'Recall ACT': 'Recall ACT (seed 42)',
                                'Precision ACT': 'Precision ACT (seed 42)', 'Forwarded': 'chuyển xử lý (seed 42)'}, what=f'B1 {s}'))
B1_PAIRS = [p for x in FC.TRAINED for p in ((f'Phẳng {x} (ngưỡng)', f'Phẳng {x}'), (f'Cascade {x}', f'Phẳng {x} (ngưỡng)'))]
B1CMP = {}
for s in ('holdout', 'lazada'):
    b = B1[s]
    B1CMP[s] = PT.pair_table(b['y'], b['diff'], b['canon'], B1_PAIRS, f'B1, {NAME[s]} (8 pairs)')
    old = rd('paper_step5_b1_comparisons.csv')
    gate_pairs(B1CMP[s], old[old['họ so sánh'] == f'B1 – {s}'], f'B1 {s}: chênh lệch, CI, McNemar = paper_step5_b1_comparisons')

ORDER_B1 = [n for x in FC.TRAINED for n in (f'Phẳng {x}', f'Phẳng {x} (ngưỡng)', f'Cascade {x}')]
RQ2_B1 = pd.concat([rows_of(B1[s]['tab'], ORDER_B1, s) for s in NAME], ignore_index=True)
B1T = rd('paper_step5_b1_validation.csv').rename(columns={'chiến lược': 'Strategy', 't': 'Threshold t', 'đạt sàn recall': 'Recall floor met on validation',
      'recall ACT': 'Validation Recall ACT', 'precision ACT': 'Validation Precision ACT', 'Macro-F1 4 lớp argmax': 'Validation Macro-F1 (argmax)',
      'Macro-F1 4 lớp có ngưỡng': 'Validation Macro-F1 (thresholded)'})
B1T['Strategy'] = B1T['Strategy'].map(PT.en)

rows = []
for s in ('holdout', 'lazada'):
    e, b = E2E[s].set_index('key'), B1[s]['tab'].set_index('key')
    bt = B1CMP[s].set_index(['key A', 'key B'])
    for x in PA.ORDER:
        f, c, t = f'Phẳng {x}', f'Cascade {x}', f'Phẳng {x} (ngưỡng)'
        r = {'Set': NAME[s], 'Strategy': PT.en(x)}
        for lab, k, src in (('Flat', f, e), ('Thresholded flat', t, b), ('Cascade', c, e)):
            if k in src.index:
                for col in ('Macro-F1', 'F1 FEATURE', 'Recall ACT', 'Precision ACT', 'Forwarded'):
                    r[f'{lab} {col}'] = src.loc[k, col]
        q = PT.pair_table(YY[s], DIFF[s], CANON[s], [(c, f)], FAM_E2E[s], holm_from=MC[s]).iloc[0]
        r.update({'Δ cascade − flat': q['Δ Macro-F1 (A − B)'], 'Δ cascade − flat CI low': q['CI low'], 'Δ cascade − flat CI high': q['CI high'],
                  'p (Holm), cascade − flat [378-pair family]': q['p (Holm)']})
        if x in FC.TRAINED:
            for lab, pair in (('thresholded − flat', (t, f)), ('cascade − thresholded', (c, t))):
                q = bt.loc[pair]
                r.update({f'Δ {lab}': q['Δ Macro-F1 (A − B)'], f'Δ {lab} CI low': q['CI low'], f'Δ {lab} CI high': q['CI high'],
                          f'p (Holm), {lab} [B1 family]': q['p (Holm)']})
        rows.append(r)
ARCH = pd.DataFrame(rows)
v = ARCH[['Set', 'Strategy', 'Flat Macro-F1', 'Thresholded flat Macro-F1', 'Cascade Macro-F1', 'Flat Recall ACT', 'Thresholded flat Recall ACT', 'Cascade Recall ACT',
          'Δ cascade − flat', 'Δ cascade − thresholded', 'Δ cascade − thresholded CI low', 'Δ cascade − thresholded CI high', 'p (Holm), cascade − thresholded [B1 family]']]
print(PT.pct(v, [c for c in v.columns if c not in ('Set', 'Strategy') and not c.startswith('p (')]).round(4).to_string(index=False))
table('rq2_architecture', ARCH, ['paper_step2_*', 'paper_step5_b1_*'], 'Architecture under the same decision policy: flat (argmax), thresholded flat and cascade of every strategy, with paired differences (95% CI) and Holm-adjusted McNemar p within the registered families.')
table('rq2_b1_configurations', RQ2_B1, ['paper_step5_b1_results.csv'], 'Full results of the flat, thresholded flat and cascade configurations of the four trained strategies (focused set descriptive).')
table('rq2_b1_thresholds', B1T, ['paper_step5_b1_validation.csv', 'paper_step5_selection.json'], 'Thresholds of the thresholded flat models chosen on validation (seed 42) with the guarded tier-1 rule.', pct=True)
B1PAIRS = pd.concat(B1CMP.values(), ignore_index=True)
table('rq2_b1_pairs', B1PAIRS, ['paper_step5_b1_comparisons.csv'], 'B1 pairwise comparisons (thresholded − flat; cascade − thresholded) with Holm within each set.')

            Set Strategy  Flat Macro-F1  Thresholded flat Macro-F1  Cascade Macro-F1  Flat Recall ACT  Thresholded flat Recall ACT  Cascade Recall ACT  Δ cascade − flat  Δ cascade − thresholded  Δ cascade − thresholded CI low  Δ cascade − thresholded CI high  p (Holm), cascade − thresholded [B1 family]
Shopee hold-out   TF-IDF           62.8                       60.8              61.3             65.6                         82.5                78.7              -1.4                      0.6                            -2.0                              3.2                                       0.0769
Shopee hold-out  Bi-LSTM           62.9                       61.7              54.2             61.7                         83.1                83.1              -8.7                     -7.5                           -12.6                             -2.3                                       0.0000
Shopee hold-out ViSoBERT           66.7                       64.9              62.9    

In [8]:
COMP = rd('paper_cost_components.csv'); RCOMP = rd('paper_step4_routing_cost_components.csv')
COMP2 = pd.concat([COMP, RCOMP], ignore_index=True)
for c in ('gọi API', 'có huấn luyện'):
    COMP2[c] = COMP2[c].astype(str).str.lower().eq('true')
CL = rd('gd10_llm_costs.csv').set_index('bước')
USD = {}
for s in ('holdout', 'lazada'):
    t1u, t2u = CL.loc[f'{s} – tầng 1', 'usd_/1000_review'] / 1000, CL.loc[f'{s} – tầng 2', 'usd_/1000_review'] / 1000
    keep = float(np.asarray(PA.canonical(PRED[s][0][PA.LLM])).mean())
    USD[s] = {'flat': cost_summary(rd(f'paper_flat_llm_{s}.csv'), prices(), s)['usd_/1000_review'] / 1000, 'cascade': t1u + keep * t2u, 'tier1': t1u}
c2 = COMP2.set_index('cấu hình')
dev = max(abs(USD['holdout']['cascade'] - c2.loc['Cascade LLM few-shot', 'USD API / review']), abs(USD['holdout']['flat'] - c2.loc['Phẳng LLM few-shot', 'USD API / review']),
          abs(USD['holdout']['tier1'] - c2.loc['LLM few-shot → PhoBERT', 'USD API / review']))
assert dev <= 1e-12
GATES.append({'check': 'USD API Hold-out của cascade LLM, LLM phẳng, LLM → PhoBERT = paper_cost_components', 'keys compared': 3, 'max abs deviation': dev})

REFS = ['Phẳng PhoBERT', 'Phẳng TF-IDF', 'Phẳng ViSoBERT', 'Phẳng LLM few-shot', 'Cascade LLM few-shot', 'LLM few-shot → PhoBERT']
RT, RT5, RCMP4, RCMP5 = {}, {}, {}, {}
for s in ('holdout', 'lazada'):
    cfg = {**{k: CFG[s][k] for k in REFS}, **R4[s]['cfg']}
    tab, diff, y, canon = PT.e2e_table(Y4[s], cfg)
    RT[s] = {'tab': tab, 'diff': diff, 'y': y, 'canon': canon}
    log_gate(f'định tuyến v4.1 {s}: Macro-F1 và CI = paper_step4_routing_{s}',
             PT.gate(tab, rd(f'paper_step4_routing_{s}.csv'), {'Macro-F1': 'Macro-F1 chuẩn', 'CI low': 'CI 2,5%', 'CI high': 'CI 97,5%'}, what=f'R4 {s}'))
    pairs4 = [(n, ref) for n in R4[s]['cfg'] for ref in ['Phẳng ' + n.split(' ⇢ ')[0], 'Phẳng LLM few-shot', 'Cascade LLM few-shot', 'LLM few-shot → PhoBERT']]
    RCMP4[s] = PT.pair_table(y, diff, canon, pairs4, f'Routing v4.1, {NAME[s]} ({len(pairs4)} pairs)')
    old = rd('paper_step4_routing_comparisons.csv')
    gate_pairs(RCMP4[s], old[old['họ so sánh'] == f'định tuyến – {s}'], f'định tuyến v4.1 {s}: chênh lệch, CI, McNemar = paper_step4_routing_comparisons')
    cfg5 = {'Phẳng PhoBERT': CFG[s]['Phẳng PhoBERT'], 'Phẳng LLM few-shot': CFG[s]['Phẳng LLM few-shot'], **R5[s]}
    tab5, diff5, y5, canon5 = PT.e2e_table(Y4[s], cfg5)
    RT5[s] = {'tab': tab5, 'diff': diff5, 'y': y5, 'canon': canon5}
    old = rd('paper_step5_b2_results.csv'); old = old[old['tập'] == s]
    log_gate(f'B2 {s}: Macro-F1 và CI = paper_step5_b2_results', PT.gate(tab5, old, {'Macro-F1': 'Macro-F1 chuẩn', 'CI low': 'CI 2,5%', 'CI high': 'CI 97,5%'}, what=f'B2 {s}'))
    pairs5 = [(PT.B2_CONF_DIS, PT.B2_DIS)] + [(r, ref) for r in (PT.B2_CONF_MAX, PT.B2_CONF_98, PT.B2_DIS) for ref in ('Phẳng PhoBERT', 'Phẳng LLM few-shot')]
    RCMP5[s] = PT.pair_table(y5, diff5, canon5, pairs5, f'B2, {NAME[s]} (7 pairs)')
    old = rd('paper_step5_b2_comparisons.csv')
    gate_pairs(RCMP5[s], old[old['họ so sánh'] == f'B2 – {s}'], f'B2 {s}: chênh lệch, CI, McNemar = paper_step5_b2_comparisons')

RND = rd('paper_step5_b2_random_baseline.csv')
def share(s, name, seed42=True):
    m = {**R4[s]['rate'], **MASK5[s]}
    if name in m:
        return float(m[name][42].mean()) if seed42 else float(np.mean([v.mean() for v in m[name].values()]))
    return 0.0 if name in ('Phẳng PhoBERT', 'Phẳng TF-IDF', 'Phẳng ViSoBERT') else 1.0

def api(s, name):
    if ' ⇢ ' in name:
        return share(s, name) * USD[s]['flat']
    return {'Phẳng LLM few-shot': USD[s]['flat'], 'Cascade LLM few-shot': USD[s]['cascade'], 'LLM few-shot → PhoBERT': USD[s]['tier1']}.get(name, 0.0)

ROUTE_KEYS = ['Phẳng PhoBERT', 'Phẳng LLM few-shot', 'Cascade LLM few-shot', 'LLM few-shot → PhoBERT', PT.B2_CONF_MAX, PT.B2_CONF_98, PT.B2_DIS, PT.B2_CONF_DIS] + \
             [f'PhoBERT ⇢ LLM (độ tin cậy, {int(round(b * 100))}%)' for b in SEL5['budgets']]
LOOK = {s: pd.concat([E2E[s], RT[s]['tab'], RT5[s]['tab']]).drop_duplicates('key').set_index('key') for s in ('holdout', 'lazada')}
LOOKC = {s: {**CANON[s], **RT[s]['canon'], **RT5[s]['canon']} for s in ('holdout', 'lazada')}
rows = []
for s in ('holdout', 'lazada'):
    rn = RND[(RND['tập'] == s) & (RND['seed'] == 42)].set_index('router')
    for k in ROUTE_KEYS:
        t = LOOK[s].loc[k]
        r = {'key': k, 'Set': NAME[s], 'Configuration': PT.en(k), 'Macro-F1': t['Macro-F1'], 'CI low': t['CI low'], 'CI high': t['CI high'],
             'F1 FEATURE': t['F1 FEATURE'], 'Precision ACT': t['Precision ACT'], 'Recall ACT': t['Recall ACT'], 'LLM share': share(s, k),
             'LLM share (3-seed mean)': share(s, k, seed42=False), 'API USD / 1,000 reviews': api(s, k) * 1000}
        if k in rn.index:
            q = rn.loc[k]
            assert abs(q['Macro-F1 router'] - t['Macro-F1']) <= 1e-12 and q['k'] == round(share(s, k) * len(Y4[s]))
            r.update({'Calls': int(q['k']), 'Random routing mean': q['random mean'], 'Random routing SD': q['random SD'], 'Random P2.5': q['random P2.5'],
                      'Random P97.5': q['random P97.5'], 'Share random ≥ router': q['share random ≥ router']})
        rows.append(r)
ROUTING = pd.DataFrame(rows)
GATES.append({'check': 'định tuyến: Macro-F1 router và số lượt gọi = paper_step5_b2_random_baseline (seed 42)', 'keys compared': int(ROUTING['Calls'].notna().sum()), 'max abs deviation': 0.0})
print(PT.pct(PT.drop_keys(ROUTING), ['Macro-F1', 'CI low', 'CI high', 'F1 FEATURE', 'Precision ACT', 'Recall ACT', 'LLM share', 'LLM share (3-seed mean)',
                                     'Random routing mean', 'Random routing SD', 'Random P2.5', 'Random P97.5']).round(3).to_string(index=False))
table('rq2_routing', ROUTING, ['paper_step4_routing_*', 'paper_step5_b2_*', 'gd10_llm_costs.csv', 'paper_flat_llm_*'], 'Routing policies to the flat LLM with PhoBERT as the small model, against the flat references and random routing with the same number of LLM calls (canonical run; random routing: 1,000 draws).')
RPAIRS = pd.concat([RCMP4['holdout'], RCMP4['lazada'], RCMP5['holdout'], RCMP5['lazada']], ignore_index=True)
table('rq2_routing_pairs', RPAIRS, ['paper_step4_routing_comparisons.csv', 'paper_step5_b2_comparisons.csv'], 'Pairwise comparisons of the routing policies; Holm within the registered v4.1 routing family (24 pairs per set) and B2 family (7 pairs per set).')
OTHER_SMALL = [k for k in R4['holdout']['cfg'] if not k.startswith('PhoBERT')]
rows = []
for s in ('holdout', 'lazada'):
    for k in ['Phẳng TF-IDF', 'Phẳng ViSoBERT'] + OTHER_SMALL:
        t = LOOK[s].loc[k]
        rows.append({'key': k, 'Set': NAME[s], 'Configuration': PT.en(k), 'Macro-F1': t['Macro-F1'], 'CI low': t['CI low'], 'CI high': t['CI high'],
                     'Precision ACT': t['Precision ACT'], 'Recall ACT': t['Recall ACT'], 'LLM share': share(s, k), 'LLM share (3-seed mean)': share(s, k, False),
                     'API USD / 1,000 reviews': api(s, k) * 1000})
table('app_routing_other_small', pd.DataFrame(rows), ['paper_step4_routing_*'], 'Supplement: confidence routing with TF-IDF and ViSoBERT as the small model.')

rows = []
for s in ('holdout', 'lazada'):
    for k in PURE + ['LLM few-shot → PhoBERT', PT.B2_CONF_MAX, PT.B2_CONF_98, PT.B2_DIS]:
        hh, ll = LOOK['holdout'].loc[k], LOOK['lazada'].loc[k]
        if s == 'holdout':
            rows.append({'key': k, 'Configuration': PT.en(k), 'Hold-out Macro-F1': hh['Macro-F1'], 'Hold-out CI low': hh['CI low'], 'Hold-out CI high': hh['CI high'],
                         'Lazada Macro-F1': ll['Macro-F1'], 'Lazada CI low': ll['CI low'], 'Lazada CI high': ll['CI high'],
                         'Δ Lazada − hold-out': ll['Macro-F1'] - hh['Macro-F1'], 'Hold-out LLM share': share('holdout', k) if ' ⇢ ' in k else np.nan,
                         'Lazada LLM share': share('lazada', k) if ' ⇢ ' in k else np.nan})
TRANSFER = pd.DataFrame(rows)
print(PT.pct(PT.drop_keys(TRANSFER), [c for c in TRANSFER.columns if c not in ('key', 'Configuration')]).to_string(index=False))
table('rq1_transfer', TRANSFER, ['paper_step2_*', 'paper_step4_routing_*', 'paper_step5_b2_*'], 'Transfer from the Shopee hold-out to Lazada: canonical-run Macro-F1 of the pure configurations, the best class-routed hybrid and the PhoBERT routers.')

            Set                                   Configuration  Macro-F1  CI low  CI high  F1 FEATURE  Precision ACT  Recall ACT  LLM share  LLM share (3-seed mean)  API USD / 1,000 reviews  Calls  Random routing mean  Random routing SD  Random P2.5  Random P97.5  Share random ≥ router
Shopee hold-out                                    Flat PhoBERT      70.8    64.6     76.0        60.9           77.6        66.1        0.0                      0.0                    0.000    NaN                  NaN                NaN          NaN           NaN                    NaN
Shopee hold-out                                        Flat LLM      80.0    75.1     84.2        69.1           70.3        90.7      100.0                    100.0                    0.266    NaN                  NaN                NaN          NaN           NaN                    NaN
Shopee hold-out                                     Cascade LLM      81.3    76.3     85.5        69.1           77.2        86.9      1

In [9]:
CAL, CALCMP = {}, {}
old_cmp = rd('paper_step4_calibration_comparisons.csv')
for s in ('holdout', 'lazada'):
    cas = {}
    for x, m in RC.TIER1.items():
        probs = RC.probabilities(RES, s, IDS[s], 'tier1', m, SEEDS)
        c = SEL4['calibration'][x]
        t1c = {k: RC.calibrate(p[:, 1], c['temperatures'][str(k)]) >= c['tau_cal'] for k, p in probs.items()}
        cas[f'Cascade {x} (hiệu chỉnh)'] = {k: cascade_labels(a, PRED[s][1][x][k]) for k, a in t1c.items()}
        cas[f'Cascade {x}'] = CFG[s][f'Cascade {x}']
    tab, diff, y, canon = PT.e2e_table(Y4[s], cas)
    CAL[s] = tab
    log_gate(f'hiệu chỉnh {s}: Macro-F1 và CI = paper_step4_calibration_cascade_{s}',
             PT.gate(tab, rd(f'paper_step4_calibration_cascade_{s}.csv'), {'Macro-F1': 'Macro-F1 chuẩn', 'CI low': 'CI 2,5%', 'CI high': 'CI 97,5%'}, what=f'cal {s}'))
    CALCMP[s] = PT.pair_table(y, diff, canon, [(f'Cascade {x} (hiệu chỉnh)', f'Cascade {x}') for x in RC.TIER1], f'Calibration, {NAME[s]} (4 pairs)')
    o = old_cmp[old_cmp['họ so sánh'] == f'hiệu chỉnh – {s}'].set_index('chiến lược')
    n = CALCMP[s].assign(x=list(RC.TIER1)).set_index('x')
    dev = max(float(np.abs(n['Δ Macro-F1 (A − B)'] * 100 - o.loc[n.index, 'chênh hiệu chỉnh − đã khóa (seed 42)']).max()),
              float(np.abs(n['p (Holm)'] - o.loc[n.index, 'p (Holm)']).max()))
    assert dev <= 1e-7
    GATES.append({'check': f'hiệu chỉnh {s}: chênh lệch và McNemar = paper_step4_calibration_comparisons', 'keys compared': len(n), 'max abs deviation': dev})
CALT = pd.concat([rows_of(CAL[s], list(CAL[s]['key']), s) for s in ('holdout', 'lazada')], ignore_index=True)
ECE = rd('paper_step4_calibration_ece.csv').rename(columns={'tập': 'Set', 'chiến lược': 'Strategy', 'ECE gốc': 'ECE before calibration', 'ECE hiệu chỉnh': 'ECE after calibration'})
ECE['Strategy'] = ECE['Strategy'].map(PT.en)
table('app_calibration', CALT, ['paper_step4_calibration_*'], 'Supplement: cascades with a temperature-scaled tier 1 against the locked cascades.')
table('app_calibration_pairs', pd.concat(CALCMP.values(), ignore_index=True), ['paper_step4_calibration_comparisons.csv'], 'Supplement: calibrated − locked cascade, Holm within each set.')
table('app_calibration_ece', ECE, ['paper_step4_calibration_ece.csv'], 'Supplement: expected calibration error of the tier-1 ACTIONABLE probability (15 equal-width bins).', pct=False)

## RQ3 — điều kiện có lợi về chi phí

**Nguồn**

Thành phần chi phí đo được (`paper_cost_components.csv`, `paper_step4_routing_cost_components.csv`); mô hình v3.8 với Macro-F1 lần chạy chuẩn; kịch bản bổ sung
của v4.2. Hòa vốn và cấu hình rẻ nhất được tính lại và phải trùng file của bước 3 và bước 4.

In [10]:
log_gate('chi phí: Macro-F1 trong thành phần chi phí = Macro-F1 lần chạy chuẩn Hold-out',
         PT.gate(LOOK['holdout'].reset_index(), COMP2, {'Macro-F1': 'Macro-F1'}, scale=100, what='cost F1'))
assert GATES[-1]['keys compared'] == len(COMP2) == 34
B = CFG_COST['base_scenario']; K = CM.drift_checks(CFG_COST); DR = CFG_COST['drift']; PR = prices()
c2 = COMP2.set_index('cấu hình')
HW = {'GPU': 'GPU', 'CPU': 'CPU', 'không': 'none (API only)'}
REP = ['Phẳng TF-IDF', 'Phẳng ViSoBERT', 'Phẳng PhoBERT', 'Cascade PhoBERT', 'Cascade LLM few-shot', 'Phẳng LLM few-shot', 'LLM few-shot → PhoBERT', PT.B2_CONF_MAX, PT.B2_CONF_98]
rows = []
for n in REP:
    r = c2.loc[n]
    fb, vb = CM.fixed_and_variable(r, CFG_COST, B['label_cost_usd'], B['gpu_usd_per_hour'], 'batch')
    fe, ve = CM.fixed_and_variable(r, CFG_COST, B['label_cost_usd'], B['gpu_usd_per_hour'], 'endpoint')
    labels = r['nhãn ban đầu'] + K * (DR['validation_labels_per_check'] + (DR['extra_train_labels_per_check'] if r['có huấn luyện'] else 0))
    rows.append({'key': n, 'Configuration': PT.en(n), 'Macro-F1 (hold-out, %)': round(r['Macro-F1'], 1), 'Hardware': HW[r['phần cứng']],
                 'Initial labels': int(r['nhãn ban đầu']), 'Labels over 12 months': int(labels), 'Tuning runs': int(r['lần chạy dò lưới']),
                 'Training seconds per run': r['giây train / lần chạy'], 'Inference seconds / 1,000 reviews': r['giây suy luận / review'] * 1000,
                 'API USD / 1,000 reviews': r['USD API / review'] * 1000, 'Fixed USD over 12 months (batch)': fb,
                 'Variable USD / 1,000 reviews (batch)': vb * 1000, 'Fixed USD over 12 months (always-on endpoint)': fe})
DRIVERS = pd.DataFrame(rows)
print(PT.drop_keys(DRIVERS).round(3).to_string(index=False))
table('rq3_cost_drivers', DRIVERS, ['paper_cost_components.csv', 'paper_step4_routing_cost_components.csv', 'paper_cost_model.yaml'], 'Measured cost drivers of representative configurations and their fixed and variable cost at USD 0.10 per label and USD 0.35 per GPU hour.', pct=False)

A_ = ['Phẳng TF-IDF', 'Phẳng ViSoBERT', 'Phẳng PhoBERT', 'Cascade PhoBERT', PT.B2_CONF_MAX, PT.B2_CONF_98]
B_ = ['Cascade LLM few-shot', 'Phẳng LLM few-shot', 'LLM few-shot → PhoBERT']
rows = []
for mode in CFG_COST['deployment']:
    for gp in CFG_COST['gpu_usd_per_hour']:
        for lc in CFG_COST['label_cost_usd']:
            for a in A_:
                for b in B_:
                    rows.append({'triển khai': mode, 'giá GPU / giờ': gp, 'chi phí một nhãn': lc, 'có huấn luyện': a, 'gọi API': b,
                                 'hòa vốn (review / tháng)': CM.break_even(COMP2, CFG_COST, a, b, lc, gp, mode)})
BE = pd.DataFrame(rows)
keys = ['triển khai', 'giá GPU / giờ', 'chi phí một nhãn', 'có huấn luyện', 'gọi API']
for f in ('paper_cost_break_even.csv', 'paper_step4_routing_break_even.csv'):
    m = BE.merge(rd(f), on=keys, suffixes=('', '_old'))
    assert len(m) > 0 and np.isclose(m['hòa vốn (review / tháng)'], m['hòa vốn (review / tháng)_old'], rtol=1e-12, atol=1e-9).all(), f
    GATES.append({'check': f'hòa vốn = {f}', 'keys compared': len(m), 'max abs deviation': 0.0})
BREAK = BE.rename(columns={'triển khai': 'Deployment', 'giá GPU / giờ': 'GPU USD per hour', 'chi phí một nhãn': 'USD per label', 'có huấn luyện': 'A (trained or routed)',
                           'gọi API': 'B (LLM)', 'hòa vốn (review / tháng)': 'Break-even reviews per month'})
BREAK['A (trained or routed)'] = BREAK['A (trained or routed)'].map(PT.en); BREAK['B (LLM)'] = BREAK['B (LLM)'].map(PT.en)
BREAK['Reading'] = BREAK['Break-even reviews per month'].map(lambda v: 'A never cheaper' if np.isinf(v) else ('A always cheaper' if v == 0 else 'A cheaper above this volume'))
v = BREAK[(BREAK['GPU USD per hour'] == B['gpu_usd_per_hour'])].pivot_table(index=['Deployment', 'A (trained or routed)', 'B (LLM)'], columns='USD per label', values='Break-even reviews per month', aggfunc='first')
print(v.round(0).to_string())
table('rq3_break_even', BREAK, ['paper_cost_break_even.csv', 'paper_step4_routing_break_even.csv'], 'Monthly volume above which configuration A becomes cheaper than the LLM configuration B over 12 months.', pct=False)

SC2 = CM.scenarios(COMP2, CFG_COST)
DEC = []
for q in CFG_COST['quality_thresholds']:
    for allow in (True, False):
        d = CM.cheapest(SC2, q, allow)
        d.insert(0, 'được gọi API', allow); d.insert(0, 'ngưỡng Macro-F1', q)
        DEC.append(d)
DEC = pd.concat([d for d in DEC if len(d)], ignore_index=True)
k2 = ['ngưỡng Macro-F1', 'được gọi API', 'triển khai', 'giá GPU / giờ', 'chi phí một nhãn', 'review / tháng']
m = DEC.merge(rd('paper_step4_cost_decision_with_routing.csv'), on=k2, suffixes=('', '_old'))
assert len(m) == len(DEC) and (m['mã'] == m['mã_old']).all() and np.allclose(m['tổng USD'], m['tổng USD_old'], rtol=1e-12)
GATES.append({'check': 'cấu hình rẻ nhất theo ngưỡng = paper_step4_cost_decision_with_routing', 'keys compared': len(m), 'max abs deviation': float(np.abs(m['tổng USD'] - m['tổng USD_old']).max())})
CHEAP = DEC.rename(columns={'ngưỡng Macro-F1': 'Macro-F1 target (%)', 'được gọi API': 'API allowed', 'triển khai': 'Deployment', 'giá GPU / giờ': 'GPU USD per hour',
                            'chi phí một nhãn': 'USD per label', 'review / tháng': 'Reviews per month', 'mã': 'Code', 'cấu hình': 'Configuration',
                            'Macro-F1': 'Macro-F1 (hold-out, %)', 'tổng USD': 'Total USD over 12 months'})
CHEAP['Configuration'] = CHEAP['Configuration'].map(PT.en); CHEAP['Code'] = CHEAP['Code'].map(PT.en_code)
v = CHEAP[(CHEAP['GPU USD per hour'] == B['gpu_usd_per_hour']) & (CHEAP['USD per label'] == B['label_cost_usd'])]
print(v.pivot_table(index=['Deployment', 'Reviews per month'], columns=['Macro-F1 target (%)', 'API allowed'], values='Code', aggfunc='first').fillna('—').to_string())
table('rq3_cheapest', CHEAP, ['paper_step4_cost_decision_with_routing.csv'], 'Cheapest configuration reaching a Macro-F1 target over 12 months in every scenario (costs within 1% or USD 10 count as equal; the higher Macro-F1 wins).', pct=False)

       Configuration  Macro-F1 (hold-out, %)        Hardware  Initial labels  Labels over 12 months  Tuning runs  Training seconds per run  Inference seconds / 1,000 reviews  API USD / 1,000 reviews  Fixed USD over 12 months (batch)  Variable USD / 1,000 reviews (batch)  Fixed USD over 12 months (always-on endpoint)
         Flat TF-IDF                    62.8             CPU            1550                   2200           18                     0.636                              0.113                    0.000                           220.000                                 0.000                                        658.000
       Flat ViSoBERT                    66.7             GPU            1550                   2200            9                    39.749                              1.393                    0.000                           220.043                                 0.000                                       3286.043
        Flat PhoBERT                    70.8  

USD per label                                        0.02       0.05       0.10       0.20       0.50
Deployment A (trained or routed) B (LLM)                                                             
batch      Cascade PhoBERT       Cascade LLM      11949.0    29930.0    59899.0   119836.0   299649.0
                                 Flat LLM          9254.0    23199.0    46441.0    92926.0   232378.0
                                 LLM → PhoBERT        0.0        0.0        0.0        0.0        0.0
           Flat PhoBERT          Cascade LLM      11946.0    29925.0    59889.0   119817.0   299601.0
                                 Flat LLM          9252.0    23196.0    46435.0    92914.0   232349.0
                                 LLM → PhoBERT        0.0        0.0        0.0        0.0        0.0
           Flat TF-IDF           Cascade LLM      11916.0    29877.0    59813.0   119685.0   299299.0
                                 Flat LLM          9231.0    23164.0    46386.0   

Macro-F1 target (%)               60              70             80
API allowed                    False   True    False    True  True 
Deployment Reviews per month                                       
batch      1000               P flat     L→L  P flat      L→L   L→L
           10000              P flat     L→L  P flat      L→L   L→L
           100000             P flat  P flat  P flat   P flat   L→L
           1000000            P flat  P flat  P flat   P flat   L→L
endpoint   1000               T flat     L→L  P flat      L→L   L→L
           10000              T flat     L→L  P flat      L→L   L→L
           100000             T flat     L→L  P flat      L→L   L→L
           1000000            T flat  T flat  P flat  T⇢L θ98   L→L


In [11]:
CODE = dict(zip(COMP2['mã'], COMP2['cấu hình'].map(PT.en)))
CODE.update({'P⇢L θ_max Laz': PT.en(PT.B2_CONF_MAX) + ' [Lazada rate]', '—': '—'})
S = rd('paper_step5_cost_target_sensitivity.csv')
SENS = pd.DataFrame({'Deployment': S['triển khai'], 'Reviews per month': S['review / tháng'], 'Macro-F1 target (%)': S['ngưỡng'], 'API allowed': S['được gọi API'],
                     'Cheapest, canonical-run Macro-F1': S['cấu hình (seed 42)'].map(CODE), 'Cheapest, three-seed mean Macro-F1': S['cấu hình (TB 3 seed)'].map(CODE)})
SENS['Choice differs'] = SENS.iloc[:, 4] != SENS.iloc[:, 5]
print('Ô có lựa chọn khác nhau giữa Macro-F1 seed 42 và trung bình ba seed:'); print(SENS[SENS['Choice differs']].to_string(index=False))
table('rq3_target_sensitivity', SENS, ['paper_step5_cost_target_sensitivity.csv'], 'Cheapest configuration for Macro-F1 targets of 78, 79 and 80 with the canonical-run Macro-F1 (main) and with the three-seed mean (sensitivity), USD 0.10 per label, USD 0.35 per GPU hour.', pct=False)

BI = rd('paper_step5_cost_existing_labels_break_even.csv')
EXIST = pd.DataFrame({'Deployment': BI['triển khai'], 'A (trained or routed)': BI['A (cố định)'].map(PT.en), 'B (LLM)': BI['B (biến đổi)'].map(PT.en),
                      'Break-even, labels purchased (reviews per month)': BI['hòa vốn gốc (review/tháng)'],
                      'Break-even, initial labels already available (reviews per month)': BI['hòa vốn khi đã có nhãn (review/tháng)']})
EXIST['Reading'] = EXIST.iloc[:, 4].map(lambda v: 'A always cheaper' if v == 0 else ('A never cheaper' if np.isinf(v) else 'A cheaper above this volume'))
table('rq3_existing_labels', EXIST, ['paper_step5_cost_existing_labels_break_even.csv', 'paper_step5_cost_existing_labels.csv'], 'Organisation that already owns the initial labels (initial labels free, drift-check labels still paid); no cheapest-configuration cell of the base grid changes (0 of 48).', pct=False)
BP = rd('paper_step5_cost_llm_price_break_even.csv')
PRICE = pd.DataFrame({'LLM price multiplier': BP['giá LLM ×'], 'Deployment': BP['triển khai'], 'A (trained or routed)': BP['A (cố định)'].map(PT.en),
                      'B (LLM)': BP['B (biến đổi)'].map(PT.en), 'Break-even reviews per month': BP['hòa vốn (review/tháng)']})
PRICE['Reading'] = PRICE['Break-even reviews per month'].map(lambda v: 'A always cheaper' if v == 0 else ('A never cheaper' if np.isinf(v) else 'A cheaper above this volume'))
table('rq3_llm_price', PRICE, ['paper_step5_cost_llm_price_break_even.csv'], 'Break-even volumes when the LLM price is multiplied by 1, 5 and 20 (USD 0.10 per label, USD 0.35 per GPU hour).', pct=False)
RR = rd('paper_step5_cost_routing_rates.csv')
RR = RR.rename(columns={'triển khai': 'Deployment', 'review / tháng': 'Reviews per month', **{c: f'{PT.en(c)}, total USD over 12 months' for c in RR.columns if c not in ('triển khai', 'review / tháng')}})
table('rq3_routing_rates', RR, ['paper_step5_cost_routing_rates.csv'], f"PhoBERT ⇢ LLM (θmax) costed with the LLM share of each test set (hold-out {share('holdout', PT.B2_CONF_MAX) * 100:.1f}%, Lazada {share('lazada', PT.B2_CONF_MAX) * 100:.1f}%, canonical run), PhoBERT inference included.", pct=False)
SO = rd('paper_step5_cost_observed_scale.csv')
SCALE = pd.DataFrame({'Configuration': SO['cấu hình'].map(PT.en), 'Macro-F1 (hold-out, %)': SO['Macro-F1 (seed 42)'].round(1), 'API USD per month': SO['USD API / tháng'],
                      'Total USD over 12 months (batch)': SO['tổng 12 tháng (USD, theo lô)'], 'of which fixed (labels and training)': SO['trong đó cố định']})
table('rq3_observed_scale', SCALE, ['paper_step5_cost_observed_scale.csv'], 'Observed collection scale: 57,200 reviews per year (about 4,767 per month), USD 0.10 per label, batch deployment; labelling and computation only, specialist labour and the cost of missed reviews are not included.', pct=False)

PARAMS = pd.DataFrame([
    ('c_L', 'Cost of one label', 'USD per label', '0.02, 0.05, 0.10 (base), 0.20, 0.50', 'assumed', 'Labelling time was not recorded; the unit price stands for single annotation. Double annotation, adjudication and audit are not costed separately.'),
    ('L_trained', 'Initial labels, trained models', 'labels', '1,550 = 1,200 Train + 350 Validation', 'measured', 'Split sizes.'),
    ('L_LLM', 'Initial labels, LLM-only configurations', 'labels', f"{CFG_COST['labels_initial']['llm_only']} = 350 Validation + 8 tier-1 + 9 tier-2 exemplars", 'measured',
     f"Counted per prompt. The 17 exemplar slots hold {int(V1['Distinct exemplars, all prompts'])} distinct Train reviews (3 appear in both tier prompts; the 14 flat-prompt exemplars are their union), so the distinct count is {L_DISTINCT}; the difference of {CFG_COST['labels_initial']['llm_only'] - L_DISTINCT} labels is USD {(CFG_COST['labels_initial']['llm_only'] - L_DISTINCT) * 0.10:.2f} at the base price and the frozen value is kept. Exemplars were drawn by a fixed seeded rule, so selection needs no further labelling effort."),
    ('k', 'Drift checks over the horizon', 'checks', str(K), 'assumed', f"One check every {DR['check_every_months']} months over {CFG_COST['horizon_months']} months."),
    ('L_check', 'Labels per drift check', 'labels', f"{DR['validation_labels_per_check']} Validation (+ {DR['extra_train_labels_per_check']} Train for trained models)", 'assumed', 'New labelled sample to re-check quality and, for trained models, to retrain.'),
    ('p_GPU', 'GPU price', 'USD per hour', ', '.join(f'{x:.2f}' for x in CFG_COST['gpu_usd_per_hour']), 'assumed', 'Cloud GPU list prices; 0.35 is the base.'),
    ('p_CPU', 'CPU price', 'USD per hour', f"{CFG_COST['cpu_usd_per_hour']:.2f}", 'assumed', 'TF-IDF training and serving.'),
    ('τ', 'Training seconds per run', 'seconds', 'per configuration', 'measured', 'Kaggle P100 GPU (neural) or local CPU (TF-IDF); grid-search runs counted.'),
    ('t', 'Inference seconds per review', 'seconds', 'per configuration', 'measured', 'Hardware-dependent; PhoBERT includes word segmentation.'),
    ('u', 'API USD per review', 'USD', f"flat LLM {USD['holdout']['flat'] * 1000:.3f} and LLM cascade {USD['holdout']['cascade'] * 1000:.3f} per 1,000 hold-out reviews", 'measured', f"Tokens returned by the API × GPT-4o-mini price (USD {PR['input']:.2f} input, {PR['output']:.2f} output per million tokens); cached input billed at the full price (upper bound)."),
    ('r', 'LLM share of a routing policy', 'share of reviews', f"θmax: hold-out {share('holdout', PT.B2_CONF_MAX) * 100:.1f}%, Lazada {share('lazada', PT.B2_CONF_MAX) * 100:.1f}% (canonical run)", 'measured', 'Taken per test set; varies by seed (see the routing table).'),
    ('e', 'Always-on endpoint', 'hours per month', str(CFG_COST['endpoint_hours_per_month']), 'assumed', 'Batch deployment pays inference time only; endpoint deployment pays the host every hour.'),
    ('V', 'Monthly review volume', 'reviews per month', ', '.join(f'{v:,}' for v in CFG_COST['monthly_volume']), 'assumed', 'Observed collection scale: 57,200 reviews per year.'),
    ('H', 'Horizon', 'months', str(CFG_COST['horizon_months']), 'assumed', ''),
    ('—', 'Not included', '', 'specialist labour, cost of missed actionable reviews, separate double annotation', 'limitation', 'Scope limited to labelling and computation.'),
], columns=['Symbol', 'Quantity', 'Unit', 'Value', 'Status (measured or assumed)', 'Source or note'])
print(PARAMS[['Symbol', 'Quantity', 'Value', 'Status (measured or assumed)']].to_string(index=False))
table('rq3_parameters', PARAMS, ['paper_cost_model.yaml', 'paper_cost_components.csv', '04_prompts/classification_fewshot/'], 'Parameters of the cost model with unit, value, status (measured or assumed) and source.', pct=False)

Ô có lựa chọn khác nhau giữa Macro-F1 seed 42 và trung bình ba seed:
Deployment  Reviews per month  Macro-F1 target (%)  API allowed Cheapest, canonical-run Macro-F1 Cheapest, three-seed mean Macro-F1  Choice differs
     batch             100000                   80         True                      Cascade LLM               PhoBERT ⇢ LLM (θmax)            True
     batch            1000000                   80         True                      Cascade LLM               PhoBERT ⇢ LLM (θmax)            True
   Symbol                                Quantity                                                                            Value Status (measured or assumed)
      c_L                       Cost of one label                                              0.02, 0.05, 0.10 (base), 0.20, 0.50                      assumed
L_trained          Initial labels, trained models                                             1,550 = 1,200 Train + 350 Validation                     measured
    L_L

## Phụ lục — theo seed, toàn bộ cấu hình, lỗi và ma trận nhầm lẫn

**Bảng theo seed**

Ghi Macro-F1 của cả ba seed, trung bình ± độ lệch chuẩn và chênh trung bình − seed 42; các ô lệch trên 2 điểm được đánh dấu để diễn giải trong bài.

In [12]:
PS = []
for s in ('holdout', 'lazada'):
    cfg = {**{k: CFG[s][k] for k in PURE if len(CFG[s][k]) > 1}, **{k: v for k, v in B1[s]['cfg'].items() if '(ngưỡng)' in k},
           **R4[s]['cfg'], **{k: v for k, v in R5[s].items() if k not in R4[s]['cfg']}}
    t = PT.per_seed_table(Y4[s], cfg)
    log_gate(f'theo seed {s}: trung bình = paper_step2_e2e_{s}_full', PT.gate(t, rd(f'paper_step2_e2e_{s}_full.csv'), {'Mean': 'Macro-F1 (trung bình)'}, what=f'seed {s}'))
    log_gate(f'theo seed {s}: trung bình = paper_step4_routing_{s}', PT.gate(t, rd(f'paper_step4_routing_{s}.csv'), {'Mean': 'Macro-F1 (trung bình)'}, what=f'seed4 {s}'))
    t.insert(1, 'Set', NAME[s])
    PS.append(t)
PER_SEED = pd.concat(PS, ignore_index=True)
print('Ô có |trung bình − seed 42| > 2 điểm:')
print(PT.pct(PT.drop_keys(PER_SEED[PER_SEED['Flag (|gap| > 2 points)']]), ['Seed 42', 'Seed 1337', 'Seed 2024', 'Mean', 'SD', 'Mean − seed 42']).to_string(index=False))
table('app_per_seed', PER_SEED, ['paper_pred_*', 'gd10_pred_*', 'paper_step4_routing_*', 'paper_step5_*'], 'Macro-F1 of every seed of the multi-run configurations, three-seed mean ± SD and the gap to the canonical run.')

PAR = rd('paper_step2_pareto.csv').set_index('cấu hình')
sc = 100 if PAR['Macro-F1'].max() > 1.5 else 1
log_gate('hình Pareto: Macro-F1 = lần chạy chuẩn', PT.gate(E2E['holdout'], PAR.reset_index(), {'Macro-F1': 'Macro-F1'}, scale=sc, what='pareto'))
ALLC = pd.concat([rows_of(E2E[s], list(CFG[s]), s, MAIN + ['Accuracy']) for s in ('holdout', 'lazada')], ignore_index=True)
ALLC['API USD / 1,000 reviews (hold-out)'] = ALLC['key'].map(lambda k: c2.loc[k, 'USD API / review'] * 1000)
ALLC['Inference seconds / 1,000 reviews'] = ALLC['key'].map(lambda k: c2.loc[k, 'giây suy luận / review'] * 1000)
ALLC['Pareto-optimal (API USD, hold-out)'] = ALLC['key'].map(PAR['Pareto (USD)'])
ALLC['Pareto-optimal (inference time, hold-out)'] = ALLC['key'].map(PAR['Pareto (giây)'])
table('app_all_configurations', ALLC, ['paper_step2_*', 'paper_cost_components.csv'], 'Supplement: all 28 configurations on both sets with measured cost and Pareto membership (Figure of quality against cost).')

ERR = pd.concat([PT.error_table(YY[s], {k: LOOKC[s][k] for k in PURE + [PT.B2_CONF_MAX, PT.B2_DIS]}).assign(Set=NAME[s]) for s in ('holdout', 'lazada')], ignore_index=True)
for _, r in ERR[ERR['Set'] == NAME['holdout']].iterrows():
    if r['key'] in ('Cascade LLM few-shot', 'Phẳng LLM few-shot', 'Phẳng TF-IDF', 'Cascade TF-IDF'):
        o = rd('paper_step2_errors_holdout.csv').set_index('cấu hình').loc[r['key']]
        assert (r['Errors'], r['Missed actionable'], r['Kept OTHER'], r['Wrong type']) == (o['tổng lỗi'], o['bỏ sót actionable'], o['giữ nhầm OTHER'], o['sai loại'])
GATES.append({'check': 'phân rã lỗi của cấu hình một lần chạy hoặc tất định = paper_step2_errors_holdout', 'keys compared': 4, 'max abs deviation': 0.0})
table('app_errors', ERR, ['paper_step2_errors_*'], 'Supplement: end-to-end errors of the canonical run split by stage.')
CONF = pd.concat([PT.confusion_table(YY[s], LOOKC[s], ['Phẳng PhoBERT', 'Cascade PhoBERT', 'Phẳng LLM few-shot', 'Cascade LLM few-shot', PT.B2_CONF_MAX]).assign(Set=NAME[s])
                  for s in ('holdout', 'lazada')], ignore_index=True)
table('app_confusion', CONF, ['paper_step2_confusion_holdout.csv'], 'Supplement: confusion matrices (gold × predicted) of the canonical runs of key configurations.', pct=False)

Ô có |trung bình − seed 42| > 2 điểm:
            Set                                   Configuration  Seed 42  Seed 1337  Seed 2024  Mean  SD  Mean − seed 42  Flag (|gap| > 2 points)
Shopee hold-out                                    Flat Bi-LSTM     62.9       60.9       55.2  59.7 4.0            -3.3                     True
Shopee hold-out                      Flat Bi-LSTM (thresholded)     61.7       51.2       43.2  52.0 9.3            -9.7                     True
         Lazada                                    Flat Bi-LSTM     54.6       51.1       46.4  50.7 4.1            -3.9                     True
         Lazada                                 Cascade Bi-LSTM     49.3       40.0       52.2  47.1 6.4            -2.1                     True
         Lazada                                 Cascade PhoBERT     56.1       53.3       48.8  52.7 3.6            -3.3                     True
         Lazada                      Flat Bi-LSTM (thresholded)     50.1       42.8   

## Hình cho bài báo

**Nội dung**

Hình định tuyến: đường Macro-F1 theo tỷ lệ gửi LLM của PhoBERT trên Hold-out (mô tả, v4.1) cùng định tuyến ngẫu nhiên cùng số lượt gọi (v4.2) và ba router đã
đăng ký. Hình chi phí: tổng chi phí 12 tháng theo khối lượng, có thêm PhoBERT ⇢ LLM (θ_max); không đọc nhãn.

In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})
FIGS = {}
CURVE = rd('paper_step4_routing_curve_holdout.csv')
cv = CURVE[CURVE['mô hình nhỏ'] == 'PhoBERT'].sort_values('llm_rate')
rk = RND[(RND['tập'] == 'holdout') & (RND['seed'] == 42)].drop_duplicates('k').sort_values('k')
rk = rk.assign(share=rk['k'] / len(Y4['holdout']))
fig, ax = plt.subplots(figsize=(6.6, 4.3))
ax.plot(cv['llm_rate'] * 100, cv['macro_f1'] * 100, color=CM.SERIES[0], lw=2, label='PhoBERT ⇢ LLM, confidence routing (θ swept)')
ax.errorbar(rk['share'] * 100, rk['random mean'] * 100,
            yerr=[(rk['random mean'] - rk['random P2.5']) * 100, (rk['random P97.5'] - rk['random mean']) * 100],
            fmt='o', color=CM.INK2, ms=3.5, lw=1, capsize=2, label='Random routing with the same number of calls (mean, 95% range)')
for k, mk, lab in ((PT.B2_CONF_MAX, 'o', 'θmax, chosen on validation'), (PT.B2_CONF_98, 's', 'θ98, chosen on validation'),
                   (PT.B2_DIS, 'D', 'PhoBERT–ViSoBERT disagreement')):
    ax.plot(share('holdout', k) * 100, LOOK['holdout'].loc[k, 'Macro-F1'] * 100, mk, color=CM.SERIES[1], ms=7, mec='white', mew=1.2, label=lab)
for k, ls, lab in (('Phẳng LLM few-shot', '--', 'Flat LLM'), ('Cascade LLM few-shot', ':', 'Cascade LLM'), ('Phẳng PhoBERT', '-.', 'Flat PhoBERT')):
    ax.axhline(LOOK['holdout'].loc[k, 'Macro-F1'] * 100, color=CM.INK2, lw=1, ls=ls, label=lab)
ax.set_xlim(-1, 45); ax.set_ylim(68, 84)
ax.set_xlabel('Reviews sent to the LLM (%)', color=CM.INK2); ax.set_ylabel('End-to-end four-class macro-F1 (%)', color=CM.INK2)
ax.grid(True, color=CM.GRID, lw=0.6); ax.set_axisbelow(True)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
for sp in ('left', 'bottom'):
    ax.spines[sp].set_color(CM.AXIS)
ax.tick_params(colors=CM.INK2)
h, l = ax.get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=2, frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.2, 1, 1))
fp = os.path.join(RES, 'paper_tables_fig_routing_en.png')
fig.savefig(fp, dpi=220, bbox_inches='tight', facecolor='white'); plt.close(fig)
FIGS['paper_tables_fig_routing_en.png'] = {'description': 'Hold-out macro-F1 of PhoBERT ⇢ LLM against the share of reviews sent to the LLM, random routing with the same number of calls and the registered routers (canonical run).',
                                           'sources': ['paper_step4_routing_curve_holdout.csv', 'paper_step5_b2_random_baseline.csv']}

NAMES = ['Phẳng TF-IDF', 'Phẳng PhoBERT', PT.B2_CONF_MAX, 'Cascade LLM few-shot', 'Phẳng LLM few-shot']
VOL = np.unique(np.round(np.logspace(3, np.log10(2e6), 60)).astype(int))
curves = CM.tco_curves(COMP2, CFG_COST, NAMES, B['label_cost_usd'], B['gpu_usd_per_hour'], VOL)
fp = os.path.join(RES, 'paper_tables_fig_cost_tco_en.png')
CM.tco_figure(curves, NAMES, {n: PT.en(n) for n in NAMES}, fp,
              {'modes': {'batch': 'Batch deployment (pay per machine hour used)', 'endpoint': 'Always-on endpoint (730 hours per month)'},
               'x': 'Reviews classified per month (log scale)', 'y': 'Total cost over 12 months (USD, log scale)'}, styles={'Phẳng TF-IDF': '--'})
FIGS['paper_tables_fig_cost_tco_en.png'] = {'description': 'Total cost over 12 months against monthly volume at USD 0.10 per label and USD 0.35 per GPU hour, batch and always-on deployment.',
                                            'sources': ['paper_cost_components.csv', 'paper_step4_routing_cost_components.csv', 'paper_cost_model.yaml']}
print('Đã lưu hình:', list(FIGS))

Đã lưu hình: ['paper_tables_fig_routing_en.png', 'paper_tables_fig_cost_tco_en.png']


## Lưu bảng, định nghĩa cột và manifest

In [14]:
PCT_ALL = {'Macro-F1', 'CI low', 'CI high', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER', 'Precision ACT', 'Recall ACT', 'F1 ACT', 'Forwarded', 'Accuracy',
           'Δ Macro-F1 (A − B)', 'Δ Lazada − hold-out', 'LLM share', 'LLM share (3-seed mean)', 'Macro-F1 (2 classes)', 'Random routing mean',
           'Random routing SD', 'Random P2.5', 'Random P97.5', 'Seed 42', 'Seed 1337', 'Seed 2024', 'Mean', 'SD', 'Mean − seed 42', 'Actionable (%)',
           'Hold-out Macro-F1', 'Hold-out CI low', 'Hold-out CI high', 'Lazada Macro-F1', 'Lazada CI low', 'Lazada CI high', 'Hold-out LLM share', 'Lazada LLM share',
           'Validation Recall ACT', 'Validation Precision ACT', 'Validation Macro-F1 (argmax)', 'Validation Macro-F1 (thresholded)',
           'Missed actionable (% of errors)', 'Kept OTHER (% of errors)', 'Wrong type (% of errors)'}
PCT_ALL |= {f'{a} {b}' for a in ('F1 ACT', 'Recall ACT', 'Precision ACT') for b in ('CI low', 'CI high')}
PCT_ALL |= {f'{lab} {col}' for lab in ('Flat', 'Thresholded flat', 'Cascade') for col in ('Macro-F1', 'F1 FEATURE', 'Recall ACT', 'Precision ACT', 'Forwarded')}
PCT_ALL |= {f'Δ {lab}{sfx}' for lab in ('cascade − flat', 'thresholded − flat', 'cascade − thresholded') for sfx in ('', ' CI low', ' CI high')}
PCT_ALL |= {c for c in DEDUP.columns if c.startswith(('Macro-F1', 'CI', 'Δ'))}

EXTRA = {
    'Set A': 'First split of the pair.', 'Set B': 'Second split of the pair.', 'Shared review IDs': 'Number of review IDs present in both splits.',
    'BUG': 'Reviews with gold label BUG.', 'FEATURE': 'Reviews with gold label FEATURE.', 'UX': 'Reviews with gold label UX.', 'OTHER': 'Reviews with gold label OTHER.',
    'Actionable': 'Reviews with gold label BUG, FEATURE or UX.', 'Actionable (%)': 'Share of actionable reviews, %.',
    'Exact duplicates of Train ∪ Validation': 'Reviews whose normalised text occurs in Train or Validation.',
    'Near duplicates (cosine ≥ 0.90)': 'Reviews without an exact match whose closest Train ∪ Validation text has character 3–5-gram TF-IDF cosine ≥ 0.90.',
    'Near duplicates (cosine ≥ 0.80)': 'Same with cosine ≥ 0.80 (sensitivity).',
    'Prompt version': 'v1 = locked prompt; v1n = app-neutral variant (Lazada only).', 'Prompt': 'tier1, tier2 or flat (four-class) prompt.',
    'Exemplars': 'Few-shot exemplars in the prompt.', 'Distinct IDs': 'Distinct review IDs among the exemplars.',
    'Gold BUG': 'Exemplars with gold label BUG.', 'Gold FEATURE': 'Exemplars with gold label FEATURE.', 'Gold UX': 'Exemplars with gold label UX.',
    'Gold OTHER': 'Exemplars with gold label OTHER.', 'All in Train': 'Every exemplar belongs to Train.', 'In another split': 'Exemplars found in any other split (must be 0).',
    'Selection seed': 'Seed of the exemplar draw.', 'Selection rule': 'Rule used to draw the exemplars.', 'Source file': 'Split the exemplars were drawn from.',
    'Distinct tier-1 + tier-2 exemplars': 'Distinct reviews over the tier-1 and tier-2 prompts.', 'Flat exemplars reused from tiers': 'Flat-prompt exemplars that also appear in a tier prompt.',
    'Flat exemplars not in tier prompts': 'Flat-prompt exemplars absent from both tier prompts.', 'Distinct exemplars, all prompts': 'Distinct reviews over all three prompts.',
    'Tier 1 or flat model': 'Model of tier 1 (cascade, hybrid), of the flat model, or the small model of a router.', 'Tier 2 model': 'Model of tier 2, or the model that relabels routed reviews.',
    'Amendment': 'Protocol amendment that registered the configuration.', 'Status': 'scored, scored as exploratory analysis, or not scored (with the reason).',
    'Macro-F1, full hold-out (n = 1,000)': 'Macro-F1 of the canonical run on the full hold-out, %.',
    'Macro-F1 without exact duplicates (n = 783)': 'Macro-F1 on the hold-out without the 217 exact duplicates of Train ∪ Validation, %.',
    'Macro-F1 without exact and near duplicates (n = 766)': 'Macro-F1 on the hold-out without exact and near (cosine ≥ 0.90) duplicates, %.',
    'CI low (without exact)': '95% bootstrap CI lower bound on the 783 reviews, %.', 'CI high (without exact)': '95% bootstrap CI upper bound on the 783 reviews, %.',
    'CI low (without exact and near)': '95% bootstrap CI lower bound on the 766 reviews, %.', 'CI high (without exact and near)': '95% bootstrap CI upper bound on the 766 reviews, %.',
    'Δ without exact': 'Macro-F1 without exact duplicates minus full hold-out, points.', 'Δ without exact and near': 'Macro-F1 without exact and near duplicates minus full hold-out, points.',
    'Macro-F1 (%)': 'Four-class Macro-F1 of the single LLM run, %.', 'F1 BUG (%)': 'F1 of BUG, %.', 'F1 FEATURE (%)': 'F1 of FEATURE, %.', 'F1 UX (%)': 'F1 of UX, %.',
    'F1 OTHER (%)': 'F1 of OTHER, %.', 'Recall ACT (%)': 'ACTIONABLE recall, %.', 'Precision ACT (%)': 'ACTIONABLE precision, %.',
    'Comparison': 'Prompt variant comparison, v1n minus v1.', 'Δ Macro-F1 (points)': 'Macro-F1 difference, percentage points.',
    'CI low (points)': '95% paired bootstrap CI lower bound of the difference, points.', 'CI high (points)': '95% paired bootstrap CI upper bound of the difference, points.',
    'Main category': 'Manually coded main topic of a gold-OTHER review labelled actionable by the LLM.',
    'Threshold t': PT.COLUMNS['Threshold t'], 'Validation Recall ACT': 'ACTIONABLE recall of the thresholded flat model on validation, seed 42, %.',
    'Validation Precision ACT': 'ACTIONABLE precision on validation, seed 42, %.', 'Validation Macro-F1 (argmax)': 'Four-class Macro-F1 on validation, flat argmax, seed 42, %.',
    'Validation Macro-F1 (thresholded)': 'Four-class Macro-F1 on validation, thresholded flat, seed 42, %.',
    'F1 ACT CI low': 'Lower 95% CI bound of F1 ACT, %.', 'F1 ACT CI high': 'Upper 95% CI bound of F1 ACT, %.', 'Recall ACT CI low': 'Lower 95% CI bound of Recall ACT, %.',
    'Recall ACT CI high': 'Upper 95% CI bound of Recall ACT, %.', 'Precision ACT CI low': 'Lower 95% CI bound of Precision ACT, %.', 'Precision ACT CI high': 'Upper 95% CI bound of Precision ACT, %.',
    'Hold-out Macro-F1': 'Canonical-run Macro-F1 on the Shopee hold-out, %.', 'Hold-out CI low': '95% CI lower bound on the hold-out, %.', 'Hold-out CI high': '95% CI upper bound on the hold-out, %.',
    'Lazada Macro-F1': 'Canonical-run Macro-F1 on Lazada, %.', 'Lazada CI low': '95% CI lower bound on Lazada, %.', 'Lazada CI high': '95% CI upper bound on Lazada, %.',
    'Hold-out LLM share': 'Share of hold-out reviews sent to the LLM (routers only), %.', 'Lazada LLM share': 'Share of Lazada reviews sent to the LLM (routers only), %.',
    'ECE before calibration': 'Expected calibration error of the raw tier-1 probability.', 'ECE after calibration': 'Expected calibration error after temperature scaling.',
    'Macro-F1 (hold-out, %)': 'Canonical-run Macro-F1 on the Shopee hold-out used by the cost model, %.', 'Hardware': 'Hardware that trains and serves the configuration.',
    'Initial labels': 'Labels needed before deployment.', 'Labels over 12 months': 'Initial labels plus the labels of the drift checks over the horizon.',
    'Tuning runs': 'Grid-search runs whose training time is paid.', 'Training seconds per run': 'Measured training time of one run, seconds.',
    'Inference seconds / 1,000 reviews': 'Measured inference time per 1,000 reviews, seconds.', 'Fixed USD over 12 months (batch)': 'Labels, training and validation API passes over 12 months, batch deployment, USD.',
    'Variable USD / 1,000 reviews (batch)': 'API and compute per 1,000 classified reviews, batch deployment, USD.',
    'Fixed USD over 12 months (always-on endpoint)': 'Fixed cost when the model is hosted on an always-on endpoint, USD.',
    'Deployment': 'batch (pay compute per review) or endpoint (always-on host).', 'GPU USD per hour': 'Assumed GPU price, USD per hour.', 'USD per label': 'Assumed price of one label, USD.',
    'A (trained or routed)': 'Configuration with the higher fixed cost.', 'B (LLM)': 'LLM configuration with the higher variable cost.',
    'Break-even reviews per month': 'Monthly volume above which A is cheaper than B over 12 months (0: A always cheaper; inf: never).', 'Reading': 'Plain reading of the break-even value.',
    'Macro-F1 target (%)': 'Quality target on the hold-out Macro-F1 (canonical run), %.', 'API allowed': 'Whether configurations that call the API may be chosen.',
    'Reviews per month': 'Monthly review volume.', 'Code': 'Short code: T = TF-IDF, B = Bi-LSTM, V = ViSoBERT, P = PhoBERT, L = LLM; X→Y tier 1 X and tier 2 Y; P⇢L routing.',
    'Total USD over 12 months': 'Total cost over 12 months, USD.', 'Cheapest, canonical-run Macro-F1': 'Cheapest configuration when the target is checked on the canonical-run Macro-F1.',
    'Cheapest, three-seed mean Macro-F1': 'Cheapest configuration when the target is checked on the three-seed mean.', 'Choice differs': 'True when the two choices differ.',
    'Break-even, labels purchased (reviews per month)': 'Break-even volume when every label is paid.',
    'Break-even, initial labels already available (reviews per month)': 'Break-even volume when the initial labels are already owned.',
    'LLM price multiplier': 'Multiplier on the measured API price.', 'API USD per month': 'API cost per month at the observed volume, USD.',
    'Total USD over 12 months (batch)': 'Total cost over 12 months at the observed volume, batch deployment, USD.', 'of which fixed (labels and training)': 'Fixed part of the total, USD.',
    'Symbol': 'Symbol of the parameter.', 'Quantity': 'Quantity the parameter stands for.', 'Unit': 'Unit.', 'Value': 'Value or grid of values.',
    'Source or note': 'Source of the value and remarks.', 'Gold': 'Gold class (row of the confusion matrix).',
    'Predicted BUG': 'Reviews predicted BUG.', 'Predicted FEATURE': 'Reviews predicted FEATURE.', 'Predicted UX': 'Reviews predicted UX.', 'Predicted OTHER': 'Reviews predicted OTHER.',
    'API USD / 1,000 reviews (hold-out)': 'Measured API cost per 1,000 hold-out reviews, USD.', 'Pareto-optimal (API USD, hold-out)': 'On the Pareto front of Macro-F1 against API cost.',
    'Pareto-optimal (inference time, hold-out)': 'On the Pareto front of Macro-F1 against inference time.', 'key A': '', 'key B': '', 'key': '',
}
for lab in ('Flat', 'Thresholded flat', 'Cascade'):
    for col in ('Macro-F1', 'F1 FEATURE', 'Recall ACT', 'Precision ACT', 'Forwarded'):
        EXTRA[f'{lab} {col}'] = f'{col} of the {lab.lower()} configuration of the strategy, canonical run, %.'
for lab, fam in (('cascade − flat', '378-pair family'), ('thresholded − flat', 'B1 family'), ('cascade − thresholded', 'B1 family')):
    EXTRA[f'Δ {lab}'] = f'Macro-F1 difference {lab}, canonical runs, points.'
    EXTRA[f'Δ {lab} CI low'] = f'Lower 95% paired bootstrap bound of Δ {lab}, points.'
    EXTRA[f'Δ {lab} CI high'] = f'Upper 95% paired bootstrap bound of Δ {lab}, points.'
    EXTRA[f'p (Holm), {lab} [{fam}]'] = f'Holm-adjusted exact McNemar p of {lab} within the {fam}.'
for c in RR.columns:
    EXTRA.setdefault(c, 'Total cost over 12 months at USD 0.10 per label and USD 0.35 per GPU hour, USD.')
EXTRA['Status (measured or assumed)'] = 'measured from the runs, assumed scenario value, or limitation.'
for c in list(HC.columns) + list(CT.columns):
    if '% of coded errors' in c:
        EXTRA.setdefault(c, 'Share of the coded false-actionable errors (gold OTHER predicted actionable) of that model and set in the category, %.')
    elif '% of OTHER reviews' in c:
        EXTRA.setdefault(c, 'False-actionable errors of the category divided by the gold-OTHER reviews of the set, %; the Total row is the false-actionable rate.')
    elif 'points of OTHER reviews' in c:
        EXTRA.setdefault(c, 'Lazada minus hold-out contribution of the category to the false-actionable rate, percentage points.')

def finalize(df, pct):
    out = PT.drop_keys(df)
    if pct:
        out = PT.pct(out, [c for c in out.columns if c in PCT_ALL])
    for c in out.columns:
        if c.startswith('p (') or c == 'Share random ≥ router':
            out[c] = out[c].astype(float).round(4)
        elif out[c].dtype == float and c not in PCT_ALL:
            out[c] = out[c].round(3)
    return out

defs, missing, manifest = [], set(), {}
for name, t in TABLES.items():
    out = finalize(t['df'], t['pct'])
    fn = f'paper_tables_{name}.csv'
    out.to_csv(os.path.join(RES, fn), index=False, encoding='utf-8-sig')
    for c in out.columns:
        d = PT.COLUMNS.get(c) or EXTRA.get(c)
        if not d:
            missing.add(c)
        defs.append({'table': fn, 'column': c, 'definition': d})
    manifest[fn] = {'description': t['note'], 'rows': len(out), 'sources': t['sources'], 'percent_columns': [c for c in out.columns if t['pct'] and c in PCT_ALL]}
assert not missing, sorted(missing)
pd.DataFrame(defs).to_csv(os.path.join(RES, 'paper_tables_columns.csv'), index=False, encoding='utf-8-sig')
pd.DataFrame(GATES).to_csv(os.path.join(RES, 'paper_tables_gates.csv'), index=False, encoding='utf-8-sig')
json.dump({'run_date': RUN_DATE, 'commit': COMMIT, 'uncommitted_changes_in_src_or_configs': DIRTY, 'python': sys.version.split()[0], 'amendment': 'v4.3',
           'notebook': 'notebooks/26_paper_tables.ipynb', 'module': '02_src/src/classification/paper_tables.py',
           'primary_estimate': 'canonical run (seed 42 for trained models, single LLM run); 95% paired percentile bootstrap, 10,000 resamples, seed 42',
           'gates_passed': len(GATES), 'tables': manifest, 'figures': FIGS}, open(os.path.join(RES, 'paper_tables_manifest.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(f'Đã lưu {len(TABLES)} bảng, {len(defs)} định nghĩa cột; {len(GATES)} cổng nhất quán đạt.')
print(pd.DataFrame(GATES).to_string(index=False))

Đã lưu 41 bảng, 409 định nghĩa cột; 42 cổng nhất quán đạt.
                                                                                       check  keys compared  max abs deviation
  trùng y hệt và gần trùng = paper_step5_near_duplicates_summary; 217 trùng y hệt ở Hold-out              9       0.000000e+00
                           end-to-end holdout: Macro-F1 và CI = paper_step2_e2e_holdout_full             28       0.000000e+00
                  end-to-end holdout: 378 chênh lệch ghép cặp = paper_step2_e2e_holdout_diff            378       9.974660e-17
                        McNemar end-to-end – Hold-out: p exact và Holm = paper_step2_mcnemar            378       1.110223e-16
                             end-to-end lazada: Macro-F1 và CI = paper_step2_e2e_lazada_full             28       5.551115e-17
                    end-to-end lazada: 378 chênh lệch ghép cặp = paper_step2_e2e_lazada_diff            378       9.887924e-17
                          McNemar end-to-end – Lazad